# 37 · Final PT pathway analysis: specificity, positional programs, robustness, replication

This notebook produces the pathway results of paper sections R2–R4 in one pass:

| Part | Paper | Question |
|---|---|---|
| E1 · Decoy-calibrated specificity | R2 | After calibrating each strategy against its own relabeled-specimen decoys, which strategies keep discoveries? Why (split-plot argument)? |
| E2 · Joint test and reporting set | R3 | Which pathways pass one joint test (covariate-matched null **and** inter-gene correlation), specimen, stability, registration and region checks? How do they group into programs? |
| E3 · External replication | R4 | Do these pathways replicate in independent multi-donor snRNA **more than pathways we did not call**? |
| E4 · Specimen-level split-plot test | R4 | With specimen × position as the error term, are gene statistics calibrated under relabeling, and do pathway calls agree? |
| E5 · Effect retention | R4 | How much of each pathway's effect survives every check, and is any region loss larger than power loss? |

**What it builds on.** The protocol is `ws2/plan.md` (workstream 2, phase 1), approved before this
notebook ran. Notebooks 12, 31 and 33 defined the model, the relabelings and the checks; this
notebook recomputes everything it needs from one coordinate. It never reads their pathway calls,
except the old 82/66/42 list, which it reads for comparison.

**Coordinate source.** By default this is notebook 13's PT scFates coordinate. If a new coordinate
is supplied (`--coordinate file.csv` with columns `structure,position`, or
`PSEUDOSPACE_PT_COORDINATE`), the whole analysis reruns on it and writes to
`results/pt_pathway_final/<label>/`.

**Cohort.** 2 control mice and 2 cortex sections from 1 human donor, all male. The section labelled
medulla is healthy cortex. Specimens are the replicates, and every result is a descriptive
within-cohort comparison.

## 1 · Folders, coordinate source and pre-specified rules

Every threshold and decision rule below was fixed in the phase-1 plan before any result in this
notebook was computed.

| Rule | Value |
|---|---|
| Discovery | joint test (matched z ÷ √VIF), positive effect, BH q ≤ 0.05 over all tested pathways |
| Specimen-sensitive | called by the joint test in either relabeling or either within-species comparison |
| Stable | retained (joint q ≤ 0.05, positive) in ≥ 5 of the 7 planned runs of notebook 12 |
| Registration check passes | effect ratio ≥ 0.7 and joint q ≤ 0.10 on the anatomy-anchored coordinate |
| Region check passes | cortical-restricted effect is positive and its ratio is not below the 5th percentile of 20 power-matched random removals of the same number of mouse S3 structures |
| Gene-subset checks pass (detected in both species ≥ 5%, mouse sex-unbiased, flagged genes removed, equal probe counts) | effect ratio ≥ 0.7 and joint q ≤ 0.10 |
| Robust | confident, and passes registration and region |
| Core | robust, and passes detected-in-both and sex-unbiased |
| Programs | average linkage on 1 − overlap coefficient of contributing members (top 10% `T_spatial`), cut 0.5 |
| E1 decision | decoy FDP is primary for R2 if `T_spatial` has ≥ 30 calls at FDP ≤ 10% and every average-level strategy has minimum FDP ≥ 0.5 |
| E2 decision | the joint-test funnel is primary if relabel joint calls ≤ 5% of species calls and ≥ 20 robust pathways remain |
| E3 decision | pathway-level replication is claimed if ≥ 50% of the robust list replicates (BH ≤ 0.10 within the list) **and** robust beats uncalled pathways (one-sided Mann–Whitney p ≤ 0.01) |
| E4 decision | replace if relabel gene p-values are calibrated (λ ≤ 1.3 and ≤ 2% of genes at p ≤ 0.01), pathway NCDR ≤ 0.13 and ≥ 30 species calls; complement if calibrated with fewer calls; drop if λ > 1.5 |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--coordinate', type=Path, help='CSV with columns structure,position')
parser.add_argument('--coordinate-label')
parser.add_argument('--probe-counts', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
coordinate_source = args.coordinate or os.environ.get('PSEUDOSPACE_PT_COORDINATE')
coordinate_file = Path(coordinate_source).expanduser() if coordinate_source else None
COORDINATE_LABEL = (args.coordinate_label or os.environ.get('PSEUDOSPACE_PT_COORDINATE_LABEL')
                    or (coordinate_file.stem if coordinate_file else 'scfates'))
probe_file = Path(args.probe_counts or os.environ.get('PSEUDOSPACE_PROBE_COUNTS')
                  or results_root / 'pt_literature_deep' / 'probe_counts.csv').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
external = data_root / 'external'
output = results_root / 'pt_pathway_final' / COORDINATE_LABEL
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            *[external / 'xiong2023_sex_bias' / f'PT-{s}_fWTvsmWT.csv' for s in ('S1', 'S2', 'S3')]]
required += [coordinate_file] if coordinate_file else []
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '37.pathway_final.1'  # Bump when a cached calculation changes.
ALPHA = .05                 # Discovery BH level.
N_NULL, SEED = 9999, 12     # Matched random sets and seed, as notebooks 12 and 31.
BASIS_DF, N_STEP_BINS = 6, 6
MIN_RETENTION = 5 / 7
DECOY_ALPHA = .10
CHECK_RATIO, CHECK_Q = .7, .10
N_POWER_DRAWS = 20
PROGRAM_CUT, CONTRIBUTING_QUANTILE = .5, .9
DETECTION_BOTH = .05
PSEUDOBULK_BINS, MIN_STRUCTURES_PER_CELL = 8, 20
MIN_CELLS_EXTERNAL = 50
FLAGGED_GENES = ['Alb', 'Ttr', 'Apoa1', 'Apoa2', 'Apoh', 'Fgb', 'Fga', 'Fgg',   # ambient hepatocyte-type
                 'Egf', 'Muc1', 'Klk1', 'Slc26a7', 'Car12', 'Akr1c18',          # neighbouring segments
                 'Ugt1a1', 'Ugt1a9']                                           # shared-exon mapping
print('Coordinate:', 'notebook 13 scFates (default)' if coordinate_file is None else coordinate_file)
print('Results folder:', output)

## 2 · Structures, coordinate, expression, gene universe and pathways

The structures, normalization, 2% equal-specimen detection rule and 10–300-gene pathway rule
match notebook 12. With the default coordinate, a guard checks that the gene universe matches
notebook 12's saved run.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.pathway_remodeling import gene_covariates, matching_strata
from pseudospace.pathways import build_pathway_membership

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
actual = pt_obs.groupby('comparison_species', observed=True)['sample'].apply(
    lambda v: sorted(v.astype(str).unique())).to_dict()
if actual != expected:
    raise ValueError(f'Unexpected cohort: {actual}')
if coordinate_file is not None:
    supplied = pd.read_csv(coordinate_file)
    if not {'structure', 'position'}.issubset(supplied) or supplied.structure.duplicated().any():
        raise ValueError('The coordinate file needs unique `structure` ids and a `position` column.')
    mapped = supplied.set_index(supplied.structure.astype(str)).position.reindex(pt_obs.index.astype(str))
    print(f'{int(mapped.notna().sum()):,} of {len(pt_obs):,} PT structures have the supplied coordinate.')
    pt_obs = pt_obs[mapped.notna().to_numpy()].copy()
    values = mapped.dropna().to_numpy(float)
    if values.min() < 0 or values.max() > 1:
        print('Supplied coordinate rescaled to [0, 1].')
        values = (values - values.min()) / (values.max() - values.min())
    pt_obs['shared_pseudospace'] = values
segment_medians = pt_obs.groupby('segment_class', observed=True).shared_pseudospace.median()
if not segment_medians['PT-S1'] < segment_medians['PT-S3']:
    raise ValueError('The coordinate must run from S1 to S3.')
support = pt_obs.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
coordinate = pt_obs.shared_pseudospace.to_numpy(float)
pt_obs = pt_obs[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad'
                for names in expected.values() for name in names}
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
position = adata.obs.shared_pseudospace.to_numpy(float)
human = adata.obs.comparison_species.astype(str).eq('human').to_numpy()
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).to_numpy()
segments = ('PT-S1', 'PT-S2', 'PT-S3')
names = expected['mouse'] + expected['human']
lo = max(position[specimen == name].min() for name in names)
hi = min(position[specimen == name].max() for name in names)
grid = np.linspace(lo, hi, 61)

expression = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)
universe = gene_covariates(expression, position, human, specimen, adata.var_names)
universe['measured_in_both'] = adata.var.measured_in_both_inputs.to_numpy()
universe['eligible'] = universe.measured_in_both & universe.detection.ge(.02)
genes = universe.loc[universe.eligible, 'gene'].tolist()
gene_index = pd.Index(genes)
eligible = universe.eligible.to_numpy()
Y = expression[:, eligible]
all_counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)
counts = all_counts[:, eligible]
library_size = np.asarray(all_counts[:, universe.measured_in_both.to_numpy()].sum(axis=1)).ravel()
covariates = universe.set_index('gene').loc[genes]
strata = matching_strata(covariates, bins=3)

libraries = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    universe.loc[universe.measured_in_both, 'gene'], ortholog_map=orthologs, library_name=library,
    tested=genes, min_genes=1) for library in libraries], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(10, min(300, len(genes) - 1))]
gene_sets = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}
pathway_names = pd.Series({p: p.split('::', 1)[1] for p in gene_sets}).str.replace(r' R-HSA-\d+$', '', regex=True)

saved12 = results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv'
saved_gene_stats = pd.read_csv(saved12, index_col=0) if coordinate_file is None and saved12.is_file() else None
if saved_gene_stats is not None and list(saved_gene_stats.index) != genes:
    raise ValueError('Default coordinate but the gene universe differs from notebook 12.')
display(adata.obs.groupby(['comparison_species', 'sample', 'segment_class'], observed=True).size()
        .unstack().rename_axis(columns=None))
print(f'{len(position):,} structures · {len(genes):,} genes · {len(gene_sets):,} pathways · '
      f'common interval {lo:.3f}–{hi:.3f}')

## 3 · Gene-set families for the gene-subset checks

Each check removes genes and keeps pathways that still have 10–300 tested members:
- **detected in both species:** at least 5% of structures in each species;
- **mouse sex-unbiased:** drops genes sex-biased in any mouse PT segment (Xiong et al. 2023
  public tables, |log2FC| ≥ 1 and padj ≤ 0.05);
- **flagged genes removed:** genes the literature check marks as ambient, neighbouring-segment
  or mapping artefacts;
- **equal probe counts:** genes with the same number of Visium HD probes in both panels, from
  workstream 3's table (`results/pt_literature_deep/probe_counts.csv`); skipped if the table is absent;
- **5% detection:** one of notebook 12's planned stability runs.

In [ ]:
def restrict(keep):
    """Gene sets restricted to `keep`, retaining pathways with 10-300 remaining members."""
    keep = set(keep)
    sets = {p: [g for g in v if g in keep] for p, v in gene_sets.items()}
    return {p: v for p, v in sets.items() if 10 <= len(v) <= min(300, len(keep) - 1)}


sex_parts = []
for seg in ('S1', 'S2', 'S3'):
    sex_table = pd.read_csv(external / 'xiong2023_sex_bias' / f'PT-{seg}_fWTvsmWT.csv').drop_duplicates('symbol').set_index('symbol')
    sex_parts.append(sex_table.p_adj.le(.05) & sex_table.log2FC.abs().ge(1))
sex_biased = pd.concat(sex_parts, axis=1).reindex(genes).fillna(False).any(axis=1)
universes = {
    'all': genes,
    'detection_5pct': covariates.index[covariates.detection.ge(.05)].tolist(),
    'detected_in_both': covariates.index[covariates.detection_mouse.ge(DETECTION_BOTH)
                                         & covariates.detection_human.ge(DETECTION_BOTH)].tolist(),
    'sex_unbiased': sex_biased.index[~sex_biased.to_numpy()].tolist(),
    'flagged_removed': [g for g in genes if g not in set(FLAGGED_GENES)],
}
probe_status = 'hook: workstream 3 probe-count table not found; probe-asymmetry check skipped'
if probe_file.is_file():
    probes = pd.read_csv(probe_file)
    # Workstream 3 (notebook 38) writes mouse_symbol, mouse_probes and human_probes (our own human run).
    probes = probes.rename(columns={'mouse_symbol': 'gene', 'mouse_probes': 'probes_mouse', 'human_probes': 'probes_human'})
    if not {'gene', 'probes_mouse', 'probes_human'}.issubset(probes):
        raise ValueError(f'{probe_file} needs columns mouse_symbol, mouse_probes, human_probes.')
    probes = probes.drop_duplicates('gene').set_index('gene').reindex(genes)
    equal_probes = probes.probes_mouse.notna() & probes.probes_mouse.eq(probes.probes_human)
    universes['probe_equal'] = equal_probes.index[equal_probes.to_numpy()].tolist()
    probe_status = f'probe-count table found: {len(universes["probe_equal"]):,} genes with equal probe counts'
families = {name: (gene_sets if name == 'all' else restrict(keep)) for name, keep in universes.items()}
print(probe_status)
display(pd.DataFrame({'genes': {k: len(v) for k, v in universes.items()},
                      'pathways': {k: len(v) for k, v in families.items()}}))

## 4 · Gene statistics for the species split, both relabelings and within-species comparisons

This is notebook 31's model. Specimen intercepts are within-group sum-to-zero contrasts, and each
specimen carries equal weight. A relabeled group holds one mouse and one human specimen, and the
species' own smooth shape enters as a nuisance term. With the default coordinate, the species
split must reproduce notebook 12's saved `T_level`, `T_spatial` and `T_total`.

In [ ]:
from pseudospace import pathway_calibration as calibration
from pseudospace import pathway_decoys as decoys_module
from pseudospace import split_plot as split_plot_module
import pseudospace.pathway_remodeling as remodeling
from pseudospace.pathway_calibration import (balanced_partitions, contrast_designs, nested_partial_f,
                                             normal_matched_q, pathway_scores, parametric_calls,
                                             negative_control_summary)
from pseudospace.pathway_remodeling import (fit_nested_trajectories, matched_pathway_tests,
                                            residual_pathway_correlations, correlation_adjusted_rank_tests,
                                            local_pathway_curves, spatial_descriptions)
from pseudospace.stats_gam import gam_internal_knots, make_gam_design
from pseudospace.stage_cache import cached_payload, cached_frame, digest

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(calibration.__file__), Path(remodeling.__file__),
               Path(decoys_module.__file__), Path(split_plot_module.__file__)])
inputs = {'Y': digest(Y), 'position': position, 'human': human, 'specimen': specimen,
          'segment': segment, 'genes': genes}
knots = gam_internal_knots(position, basis_df=BASIS_DF)
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))
swaps = [label for label in partitions if label != 'species']
step_bins = np.clip(((position - lo) / (hi - lo) * N_STEP_BINS).astype(int), 0, N_STEP_BINS - 1).astype(str)
print('Partitions:', partitions)


def group_of(label):
    if label == 'species':
        return human.astype(float), None
    return np.isin(specimen, partitions[label]).astype(float), human.astype(float)


def partition_scores(label):
    group, nuisance = group_of(label)
    designs, weights = contrast_designs(position, group, specimen, knots, segments=segment, nuisance_shape=nuisance)
    scores = nested_partial_f(Y, designs, weights)
    designs, weights = contrast_designs(position, group, specimen, knots, segments=step_bins, nuisance_shape=nuisance)
    binned = nested_partial_f(Y, designs, weights, {'T_bins_spatial': ('level', 'discrete_shape')})
    scores.pop('df'), binned.pop('df')
    return {**scores, **binned}


def within_species_scores(pair):
    keep = np.isin(specimen, pair)
    designs, weights = contrast_designs(position[keep], (specimen[keep] == pair[0]).astype(float), specimen[keep],
                                        gam_internal_knots(position[keep], basis_df=BASIS_DF))
    scores = nested_partial_f(Y[keep], designs, weights, {'T_level': ('base', 'level'), 'T_spatial': ('level', 'full')})
    scores.pop('df')
    return scores


def stage_name(prefix, label):
    return prefix + label.replace(':', '_').replace('+', '_')


gene_scores = {}
for label in partitions:
    gene_scores[label] = pd.DataFrame(cached_payload(stage_name('gene_scores_', label),
        lambda label=label: partition_scores(label), root=cache,
        params={'basis_df': BASIS_DF, 'n_bins': N_STEP_BINS, 'label': label}, inputs=inputs, code=code), index=genes)
for species_name, pair in expected.items():
    gene_scores['within_' + species_name] = pd.DataFrame(cached_payload('gene_scores_within_' + species_name,
        lambda pair=pair: within_species_scores(pair), root=cache, params={'basis_df': BASIS_DF, 'pair': pair},
        inputs=inputs, code=code), index=genes)
if saved_gene_stats is not None:  # Guard, every run with the default coordinate.
    for column in ('T_level', 'T_spatial', 'T_total'):
        np.testing.assert_allclose(gene_scores['species'][column], saved_gene_stats[column], rtol=1e-6, atol=1e-8)
T_spatial = gene_scores['species'].T_spatial
display(pd.DataFrame({k: v.median() for k, v in gene_scores.items()}).T.round(2).rename_axis('median gene score'))

## 5 · Species fit, fitted curves and residual correlations

The full species model gives the fitted human and mouse curves, Δ(s) with HC3 standard errors, and
residuals. Mean residual correlations among each pathway's genes are estimated within every
specimen, then Fisher-averaged with equal weights, as in notebook 12. They give each pathway's
CAMERA variance inflation (VIF), for every gene-set family. Only the correlations are cached, not
the 12,000 × 11,000 residual matrix.

In [ ]:
_held = {}


def species_fit():
    if 'fit' not in _held:
        _held['fit'] = fit_nested_trajectories(Y, position, human, specimen, grid, basis_df=BASIS_DF,
                                               return_residuals=True)
    return _held['fit']


fit = cached_payload('species_fit', lambda: {k: v for k, v in species_fit().items() if k != 'residuals'},
                     root=cache, params={'basis_df': BASIS_DF, 'grid': grid}, inputs=inputs, code=code)
correlations = cached_frame('residual_correlations', lambda: pd.concat(
    [residual_pathway_correlations(species_fit()['residuals'], specimen, genes, sets).assign(family=name)
     for name, sets in families.items()], ignore_index=True),
    root=cache, params={'basis_df': BASIS_DF}, inputs={**inputs, 'families': families}, code=code)
_held.clear()
np.testing.assert_allclose(fit['T_spatial'], T_spatial, rtol=1e-6, atol=1e-8)
vif = {}
for name, sets in families.items():
    family_genes = universes[name]
    camera = correlation_adjusted_rank_tests(T_spatial.loc[family_genes], sets,
                                             correlations[correlations.family.eq(name)].drop(columns='family'))
    vif[name] = camera.set_index('pathway_id').variance_inflation
saved_camera = results_root / 'pt_pathway_remodeling_scfates' / 'pathway_spatial_correlation_tests.csv'
if saved_gene_stats is not None and saved_camera.is_file():  # Guard: same inflation as notebook 12.
    reference = pd.read_csv(saved_camera).set_index('pathway_id').variance_inflation
    np.testing.assert_allclose(vif['all'].reindex(reference.index), reference, rtol=1e-6)
delta = pd.DataFrame(fit['delta'], index=genes, columns=np.round(grid, 4))
zgrid = pd.DataFrame(fit['z'], index=genes, columns=np.round(grid, 4))
print('Median VIF by family:', {k: round(float(v.median()), 2) for k, v in vif.items()})

## 6 · Matched pathway tests on every partition, and the joint test

Each test is a one-sided rank-AUC against equally sized random gene sets drawn within notebook
12's covariate strata (9,999 draws). The matched null controls measurability but treats genes as
independent. The **joint test** divides the matched z by √VIF, so the variance also reflects
inter-gene correlation, then applies BH within each partition and statistic. The same species
VIF is used for every partition: residual correlation is a property of the genes, not of the
contrast.

In [ ]:
from pseudospace.pathway_decoys import (joint_matched_test, decoy_fdp, decoy_calls, bootstrap_decoy_calls,
                                        decoy_inflation, overlap_programs, averaging_loss, shape_class, welch_t)


def matched(stage, frame, sets, strata_values):
    return cached_frame(stage, lambda: matched_pathway_tests(frame, sets, strata_values, n_null=N_NULL, seed=SEED),
                        root=cache, params={'n_null': N_NULL, 'seed': SEED},
                        inputs={'scores': frame, 'sets': sets, 'strata': strata_values}, code=code)


MAIN_STATISTICS = ['T_level', 'T_spatial', 'T_total', 'T_discrete_spatial', 'T_bins_spatial', 'T_position_given_segments']
matched_parts = []
for label, frame in gene_scores.items():
    columns = ['T_level', 'T_spatial'] if label.startswith('within_') else MAIN_STATISTICS
    matched_parts.append(matched(stage_name('matched_', label), frame[columns], gene_sets, strata).assign(partition=label))
tests = joint_matched_test(normal_matched_q(pd.concat(matched_parts, ignore_index=True)), vif['all'])
uniform = joint_matched_test(normal_matched_q(pd.concat(
    [matched(stage_name('uniform_', label), gene_scores[label][['T_spatial']], gene_sets,
             np.zeros(len(genes), int)).assign(partition=label) for label in partitions], ignore_index=True)), vif['all'])
spatial = tests[tests.statistic.eq('T_spatial')]
species_spatial = spatial[spatial.partition.eq('species')].set_index('pathway_id')
if saved_gene_stats is not None:
    atlas12 = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'pathway_evidence_atlas.csv').set_index('pathway_id')
    np.testing.assert_allclose(species_spatial.loc[atlas12.index, 'auc'], .5 + atlas12.effect_T_spatial, atol=1e-6)
tests.to_csv(tables / 'matched_and_joint_tests_all_partitions.csv', index=False)
call_counts = tests.assign(
    matched=lambda d: d.effect.gt(0) & d.q_normal.le(ALPHA),
    joint=lambda d: d.effect.gt(0) & d.q_joint.le(ALPHA)).pivot_table(
    index='statistic', columns='partition', values=['matched', 'joint'], aggfunc='sum')
display(call_counts)

## 7 · E1 · Why positional screens are specific: the split-plot argument

**Model.** Take one gene in specimen j ∈ {1, 2} of species i ∈ {M, H} at PT position s:

$$y_{ij}(s) = \mu(s) + \alpha_i + \beta_i(s) + a_{ij} + b_{ij}(s) + \varepsilon,$$

- α_i and β_i(s) are the species offset and species shape.
- a_ij is a specimen offset.
- b_ij(s) is a specimen-specific positional deviation with mean zero over s.
- a and b are independent across specimens, with variances σ²_{a,i} and σ²_{b,i} that may differ
  by species.

Species is applied to whole specimens and position varies within each specimen. This is a
**split-plot** layout: specimens are whole plots and positions are subplots.

**Error terms.**
- The species offset estimate is ½(ȳ_H1 + ȳ_H2 − ȳ_M1 − ȳ_M2). It carries
  ½(a_H1 + a_H2 − a_M1 − a_M2), and the only replicate information on that noise is the two
  within-species differences a_H1 − a_H2 and a_M1 − a_M2: **2 df**.
- The species-by-position estimate carries ½(b_H1 + b_H2 − b_M1 − b_M2)(s). Its noise is estimated
  from b_H1 − b_H2 and b_M1 − b_M2 at every position: **2(k − 1) df** with k position bins or basis
  functions.

An offset screen therefore rests on two numbers per gene, while a positional screen averages
specimen deviations over many positions.

**The relabelings are the other two orthogonal contrasts.** Four specimens form exactly three
balanced 2 + 2 splits:
- species: ½(H1 + H2 − M1 − M2);
- relabeling 1: ½(H1 + M1 − H2 − M2);
- relabeling 2: ½(H1 + M2 − H2 − M1).

Every specimen enters each contrast with weight ±½. Under independent specimen deviations, all
three contrasts carry noise of the same variance, for offsets and for shapes. Each relabeled
screen is therefore one draw from the species screen's **section-level** null.

**What relabeling cannot see.** Some components are shared by both human sections: a donor
effect d_H(s), or anything common to the human probe panel, sampled region or coordinate
registration. Such a component enters the species contrast fully and cancels in both relabelings.
Relabeling therefore measures section-level specificity only. The registration and region checks
(E2) and external donors (E3) address the rest.

**What two decoys can and cannot support.**
- Per pathway: only two null draws exist, so a per-pathway test has 2 df and cannot survive
  multiple testing (checked below).
- Pooled over 1,513 pathways: the decoys estimate a whole screen's false-discovery proportion. Of
  the pathways a screen calls at threshold t, an estimated FDP(t) = mean decoy calls ÷ species
  calls are specimen noise (target–decoy; Elias & Gygi 2007). The NCDR of notebook 31 is this
  FDP at the screen's own BH threshold.
- The decoys also estimate how much specimen-level correlation inflates matched z:
  E[z²] = a·(1 + (m − 1)ρ) for a set of m genes.

In [ ]:
import gseapy as gp
from importlib.metadata import version
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm, chi2, t as t_dist, spearmanr, mannwhitneyu
from pseudospace.conventional_pt import deseq2_contrast

blocks = {label: pd.DataFrame(np.vstack([np.asarray(counts[(specimen == n) & (segment == label)].sum(axis=0)).ravel()
                                         for n in names]), index=names, columns=genes) for label in segments}
count_support = np.logical_and.reduce([blocks[s].ge(10).sum(axis=0).ge(2).to_numpy() for s in segments])
count_genes = pd.Index(genes)[count_support]
count_sets = {p: [g for g in v if g in set(count_genes)] for p, v in gene_sets.items()}
count_sets = {p: v for p, v in count_sets.items() if 10 <= len(v) <= 300}
whole_counts = sum(blocks[s] for s in segments)[count_genes].astype(np.int64)
conventional_code = digest([code, 'conventional-v1', version('pydeseq2'), version('gseapy')])


def prerank(stat, sets, weight):
    ranking = stat.dropna().rename_axis('gene').reset_index(name='stat').sort_values(
        ['stat', 'gene'], ascending=[False, True], kind='stable')
    table = gp.prerank(rnk=ranking, gene_sets=sets, min_size=10, max_size=300, weight=weight,
                       method='multilevel', seed=SEED, threads=4, outdir=None, no_plot=True).res2d
    table = table.rename(columns={'Term': 'pathway_id'})
    return pd.DataFrame({'pathway_id': table.pathway_id, 'NES': pd.to_numeric(table.NES),
                         'p': pd.to_numeric(table['NOM p-val'])})


def conventional_deseq2(label):
    metadata = pd.DataFrame({'species': ['human' if n in expected['human'] else 'mouse' for n in names]}, index=names)
    design, contrast = '~species', ('species', 'human', 'mouse')
    if label != 'species':
        metadata['group'] = ['a' if n in partitions[label] else 'b' for n in names]
        design, contrast = '~species + group', ('group', 'a', 'b')
    scopes = [('whole_PT', whole_counts)] + [(s, blocks[s][count_genes].astype(np.int64)) for s in segments]
    return pd.concat([deseq2_contrast(block, metadata, design=design, contrast=contrast).assign(scope=scope)
                      for scope, block in scopes], ignore_index=True)


count_inputs = {'counts': [whole_counts] + [blocks[s][count_genes] for s in segments], 'sets': count_sets}
conventional_de, conventional_gsea = [], []
for label in partitions:
    de = cached_frame(stage_name('deseq2_', label), lambda label=label: conventional_deseq2(label), root=cache,
                      params={'label': label}, inputs=count_inputs, code=conventional_code)
    gsea = cached_frame(stage_name('signed_gsea_', label),
        lambda de=de: pd.concat([prerank(rows.set_index('gene').stat, count_sets, 1).assign(scope=scope)
                                 for scope, rows in de.groupby('scope', sort=True)], ignore_index=True),
        root=cache, params={'weight': 1, 'method': 'multilevel', 'seed': SEED},
        inputs={'de': de[['gene', 'stat', 'scope']], 'sets': count_sets}, code=conventional_code)
    conventional_de.append(de.assign(partition=label))
    conventional_gsea.append(gsea.assign(partition=label))
conventional_de = pd.concat(conventional_de, ignore_index=True)
conventional_gsea = pd.concat(conventional_gsea, ignore_index=True)
conventional_gsea['family'] = np.where(conventional_gsea.scope.eq('whole_PT'), 'whole_PT', 'segments')
conventional_gsea['q'] = conventional_gsea.groupby(['partition', 'family']).p.transform(
    lambda p: multipletests(p, method='fdr_bh')[1])
spatial_gsea = pd.concat([cached_frame(stage_name('spatial_gsea_', label),
    lambda label=label: prerank(gene_scores[label].T_spatial, gene_sets, 0), root=cache,
    params={'weight': 0, 'method': 'multilevel', 'seed': SEED},
    inputs={'scores': gene_scores[label].T_spatial, 'sets': gene_sets}, code=conventional_code).assign(partition=label)
    for label in partitions], ignore_index=True)
spatial_gsea['q'] = spatial_gsea.groupby('partition').p.transform(lambda p: multipletests(p, method='fdr_bh')[1])
score_parts = []
for label in partitions:
    group, nuisance = group_of(label)
    designs, weights = contrast_designs(position, group, specimen, knots, segments=segment, nuisance_shape=nuisance)
    module_scores = pathway_scores(Y, gene_sets, genes, weights)
    result = nested_partial_f(module_scores.to_numpy(), designs, weights,
                              {'T_level': ('base', 'level'), 'T_spatial': ('level', 'full')})
    df = result.pop('df')
    score_parts.append(parametric_calls({k: pd.Series(v, index=module_scores.columns) for k, v in result.items()},
                                        df, ALPHA).assign(partition=label))
score_calls = pd.concat(score_parts, ignore_index=True)
de_counts = conventional_de.assign(deg=lambda d: d.padj.le(ALPHA)).groupby(['partition', 'scope']).deg.sum().unstack()
display(de_counts.rename_axis(columns='DESeq2 genes, padj ≤ 0.05'))

### Strategy table: BH calls, NCDR and decoy-calibrated calls

Every strategy is scored for the species split and both relabelings on the same pathways:

| Strategy | Score used for decoy calibration |
|---|---|
| Gene-GAM strategies | matched z, or joint z for the joint versions |
| Size-only null | its z |
| Preranked GSEA on `T_spatial` | NES |
| Signed whole-PT GSEA | \|NES\| |
| Segment GSEA | max \|NES\| across segments |
| Pathway-score models | F |

In [ ]:
strategy_rows, decoy_rows, decoy_curves = [], [], {}


def add_strategy(name, family, frame, score_column, hit):
    """frame: rows per (partition, pathway_id) with the score; hit: boolean Series aligned to frame."""
    scores = frame.pivot_table(index='pathway_id', columns='partition', values=score_column, aggfunc='first')
    scores = scores[['species', *swaps]].dropna()
    hits = frame.assign(hit=hit.to_numpy()).pivot_table(index='pathway_id', columns='partition', values='hit',
                                                        aggfunc='first').reindex(scores.index).fillna(False).astype(bool)
    curve = decoy_fdp(scores['species'], scores[swaps])
    decoy_curves[name] = curve
    calls = decoy_calls(scores['species'], scores[swaps], DECOY_ALPHA)
    interval = bootstrap_decoy_calls(scores['species'], scores[swaps], DECOY_ALPHA, n_boot=300, seed=SEED)
    species_calls, null_calls = int(hits['species'].sum()), [int(hits[s].sum()) for s in swaps]
    strategy_rows.append({'strategy': name, 'family': family, 'tested': len(scores), 'species_calls': species_calls,
                          **{'relabel_calls_' + s.split('+')[-1]: n for s, n in zip(swaps, null_calls)},
                          'ncdr': np.mean(null_calls) / species_calls if species_calls else np.nan,
                          'null_rate': np.mean(null_calls) / len(scores),
                          'decoy_calls_fdp10': len(calls), 'decoy_calls_ci_low': interval[0],
                          'decoy_calls_ci_high': interval[2],
                          'min_fdp_10plus': float(curve.loc[curve.target_calls >= 10, 'fdp'].min())})


whole = conventional_gsea[conventional_gsea.scope.eq('whole_PT')].assign(abs_nes=lambda d: d.NES.abs())
add_strategy('1 · Whole-PT pseudobulk + signed GSEA', 'average', whole, 'abs_nes', whole.q.le(ALPHA))
seg = (conventional_gsea[conventional_gsea.family.eq('segments')].assign(abs_nes=lambda d: d.NES.abs())
       .groupby(['partition', 'pathway_id']).agg(abs_nes=('abs_nes', 'max'), q=('q', 'min')).reset_index())
add_strategy('2 · S1/S2/S3 pseudobulk + signed GSEA', 'average', seg, 'abs_nes', seg.q.le(ALPHA))
for stat, label, family in (('T_level', '3a · Pathway-score GAM, offset', 'average'),
                            ('T_spatial', '3b · Pathway-score GAM, shape', 'position, structure-level')):
    rows = score_calls[score_calls.statistic.eq(stat)]
    add_strategy(label, family, rows, 'F', rows.hit)
labels = {'T_level': ('4 · Gene GAM offset (T_level)', 'average'), 'T_total': ('5 · Gene GAM total (T_total)', 'average'),
          'T_discrete_spatial': ('6 · Gene GAM × S1/S2/S3 steps', 'position, steps'),
          'T_bins_spatial': (f'7 · Gene GAM × {N_STEP_BINS} equal bins', 'position, steps'),
          'T_spatial': ('8 · Gene GAM × smooth position (T_spatial)', 'position, smooth'),
          'T_position_given_segments': ('Gene GAM: smooth position beyond S1/S2/S3 steps', 'position, smooth')}
for stat, (label, family) in labels.items():
    rows = tests[tests.statistic.eq(stat) & tests.partition.isin(list(partitions))]
    add_strategy(label, family, rows, 'z_matched', rows.effect.gt(0) & rows.q_normal.le(ALPHA))
    add_strategy(label + ' (joint test)', family, rows, 'z_joint', rows.effect.gt(0) & rows.q_joint.le(ALPHA))
rows = uniform
add_strategy('8b · T_spatial, size-only random sets', 'position, smooth', rows, 'z_matched',
             rows.effect.gt(0) & rows.q_normal.le(ALPHA))
add_strategy('8c · T_spatial, preranked GSEA', 'position, smooth', spatial_gsea, 'NES',
             spatial_gsea.NES.gt(0) & spatial_gsea.q.le(ALPHA))
specificity = pd.DataFrame(strategy_rows).set_index('strategy')
specificity['specific_nb31_rule'] = specificity.ncdr.lt(.25) & specificity.null_rate.lt(.05)
specificity.to_csv(tables / 'table_S1_strategy_specificity.csv')
display(specificity.round(3))

average_min_fdp = specificity.loc[specificity.family.eq('average'), 'min_fdp_10plus'].min()
spatial_decoy_calls = int(specificity.loc['8 · Gene GAM × smooth position (T_spatial) (joint test)', 'decoy_calls_fdp10'])
E1_PRIMARY = spatial_decoy_calls >= 30 and average_min_fdp >= .5
print(f'E1 decision: joint T_spatial calls at decoy FDP ≤ 10% = {spatial_decoy_calls}; '
      f'lowest FDP of any average-level strategy = {average_min_fdp:.2f} → '
      + ('decoy FDP is the primary R2 metric.' if E1_PRIMARY else 'keep NCDR as primary and state the limitation.'))

### Reading E1

The pre-specified rule requires every average-level strategy to have minimum FDP ≥ 0.5. The
pathway-score models break it, and the reason matters for R2.

- **Their scores are self-contained:** each pathway's mean score is compared between groups.
- **For the species contrast, nearly every pathway's mean score differs** by more than any
  section-to-section difference, because thousands of genes differ between species (DESeq2,
  above).
- **So their decoy FDP is near zero, yet they call almost every tested pathway**, for species and
  for relabelings alike (null rate in the table). The decoy FDP shows only that these differences
  exceed section-level noise. It cannot separate species biology from anything else the human
  sections share (donor, probe panel, region), and it does not discriminate between pathways.

Following the pre-specified rule, R2 keeps notebook 31's specificity rule (NCDR plus the null-rate
condition) as its primary metric. The decoy FDP is reported as a secondary metric for
**competitive** strategies, where it does discriminate: compare the minimum FDP of the
competitive average-level screens with that of the position-dependent screens in the table above.

In [ ]:
inflation_rows = []
for stat in MAIN_STATISTICS:
    z = tests[tests.statistic.eq(stat) & tests.partition.isin(swaps)].pivot(index='pathway_id', columns='partition', values='z_matched')
    sizes = species_spatial.n_genes.reindex(z.index)
    a, rho, _ = decoy_inflation(z, sizes)
    relabel_rows = tests[tests.statistic.eq(stat) & tests.partition.isin(swaps)]
    nominal = (relabel_rows.effect.gt(0) & relabel_rows.p_normal.le(.01)).groupby(relabel_rows.partition).sum()
    inflation_rows.append({'statistic': stat, 'a': a, 'rho': rho, 'nominal_p01_relabel_mean': float(nominal.mean()),
                           'nominal_p01_expected': .01 * len(z)})
inflation = pd.DataFrame(inflation_rows).set_index('statistic')
inflation.to_csv(tables / 'decoy_inflation_by_statistic.csv')
display(inflation.round(4))
null_z = spatial[spatial.partition.isin(swaps)].pivot(index='pathway_id', columns='partition', values='z_matched')
per_pathway_t = species_spatial.z_matched / np.sqrt((null_z ** 2).mean(axis=1)).reindex(species_spatial.index)
per_pathway_q = multipletests(t_dist.sf(per_pathway_t, 2), method='fdr_bh')[1]
print(f'Per-pathway test with the two decoys as error (2 df): {int((per_pathway_q <= ALPHA).sum())} calls; '
      f'smallest p {t_dist.sf(per_pathway_t, 2).min():.1e}. Per-pathway specimen-level inference is not possible.')

## 8 · E2 · The joint-test funnel

1. **Candidates.** Joint test on `T_spatial`, positive effect, q ≤ 0.05.
2. **Not specimen-sensitive.** Not called by the joint test in either relabeling or either
   within-species comparison.
3. **Stable.** Retained in ≥ 5 of notebook 12's 7 planned runs: each specimen left out, basis 4
   and 8, and 5% detection. Every run refits all genes and uses the joint test.
4. **Robust.** Passes the registration check and the power-adjusted region check.
5. **Core.** Robust, and passes the detected-in-both and mouse sex-unbiased checks.

In [ ]:
def spatial_scores(rows, pos, gene_mask=None, basis_df=BASIS_DF):
    """T_spatial for the species split on a subset of structures (and genes), knots from that subset."""
    designs, weights = contrast_designs(pos[rows], human[rows].astype(float), specimen[rows],
                                        gam_internal_knots(pos[rows], basis_df=basis_df))
    y = Y[rows] if gene_mask is None else Y[rows][:, gene_mask]
    return nested_partial_f(y, designs, weights, {'T_spatial': ('level', 'full')})['T_spatial']


def joint_spatial(stage, scores, family='all', strata_values=None, sets=None):
    """Matched + joint test of one T_spatial vector over a gene-set family."""
    family_genes = pd.Index(universes[family])
    sets = families[family] if sets is None else sets
    strata_values = (pd.Series(strata, index=genes).loc[family_genes].to_numpy() if strata_values is None
                     else strata_values)
    frame = pd.DataFrame({'T_spatial': scores.loc[family_genes]})
    return joint_matched_test(matched(stage, frame, sets, strata_values), vif[family]).set_index('pathway_id')


candidates = set(species_spatial.index[species_spatial.effect.gt(0) & species_spatial.q_joint.le(ALPHA)])
null_spatial = spatial[~spatial.partition.eq('species')]
sensitive = set(null_spatial.loc[null_spatial.effect.gt(0) & null_spatial.q_joint.le(ALPHA), 'pathway_id'])

all_rows = np.ones(len(position), bool)
variants = [(f'omit_{name}', specimen != name, BASIS_DF, .02, 'all') for name in names]
variants += [('basis_4', all_rows, 4, .02, 'all'), ('basis_8', all_rows, 8, .02, 'all'),
             ('detection_5pct', all_rows, BASIS_DF, .05, 'detection_5pct')]
stability_parts = []
for label, keep_rows, df, detection_cut, family in variants:
    keep_genes = covariates.detection.ge(detection_cut).to_numpy()
    variant_genes = gene_index[keep_genes]
    variant_inputs = {**inputs, 'keep_rows': keep_rows, 'keep_genes': keep_genes, 'basis_df': df}
    scores = pd.Series(cached_payload(label + '_spatial', lambda keep_rows=keep_rows, keep_genes=keep_genes, df=df:
                                      {'T_spatial': spatial_scores(keep_rows, position, keep_genes, df)},
                                      root=cache, params={'basis_df': df}, inputs=variant_inputs, code=code)['T_spatial'],
                       index=variant_genes)
    variant_strata = matching_strata(gene_covariates(Y[keep_rows][:, keep_genes], position[keep_rows], human[keep_rows],
                                                     specimen[keep_rows], variant_genes), bins=3)
    result = joint_matched_test(matched(label + '_matched', pd.DataFrame({'T_spatial': scores}), families[family],
                                        variant_strata), vif[family]).set_index('pathway_id')
    run = result.reindex(sorted(candidates))
    stability_parts.append(pd.DataFrame({'pathway_id': run.index, 'variant': label, 'testable': run.q_joint.notna().to_numpy(),
                                         'retained': (run.effect.gt(0) & run.q_joint.le(ALPHA)).to_numpy(),
                                         'effect': run.effect.to_numpy(), 'q_joint': run.q_joint.to_numpy()}))
stability = pd.concat(stability_parts, ignore_index=True)
stability.to_csv(tables / 'stability_runs.csv', index=False)
stability_summary = stability.groupby('pathway_id').agg(n_testable=('testable', 'sum'), n_retained=('retained', 'sum'))
stability_summary['retention_fraction'] = stability_summary.n_retained / stability_summary.n_testable.replace(0, np.nan)
stable = set(stability_summary.index[stability_summary.retention_fraction.ge(MIN_RETENTION - 1e-9)])
confident = (candidates - sensitive) & stable
print(f'Candidates {len(candidates)} → not specimen-sensitive {len(candidates - sensitive)} → stable {len(confident)}')

### Registration: anatomy-anchored human coordinate

The reviewed S1→S2 and S2→S3 transitions are estimated per specimen by logistic regression on
position. Each human section is warped piecewise-linearly onto the mouse transitions, then every
gene is refit and the pathway screen rerun. A result that comes from segment-level
misregistration should disappear.

In [ ]:
from sklearn.linear_model import LogisticRegression

transition_rows = []
for name in names:
    for first, second in (('PT-S1', 'PT-S2'), ('PT-S2', 'PT-S3')):
        keep = (specimen == name) & np.isin(segment, [first, second])
        model = LogisticRegression(C=1e4).fit(position[keep][:, None], (segment[keep] == second).astype(int))
        transition_rows.append({'specimen': name, 'species': 'human' if name in expected['human'] else 'mouse',
                                'transition': f'{first[-2:]}→{second[-2:]}',
                                'position': float(-model.intercept_[0] / model.coef_[0, 0])})
transitions = pd.DataFrame(transition_rows)
transitions.to_csv(tables / 'segment_transitions_by_specimen.csv', index=False)
mouse_anchor = transitions[transitions.species.eq('mouse')].groupby('transition').position.mean()
warped = position.copy()
for name in expected['human']:
    own = transitions[transitions.specimen.eq(name)].set_index('transition').position
    knots_from = np.r_[0., own['S1→S2'], own['S2→S3'], 1.]
    knots_to = np.r_[0., mouse_anchor['S1→S2'], mouse_anchor['S2→S3'], 1.]
    if np.any(np.diff(knots_from) <= 0) or np.any(np.diff(knots_to) <= 0):
        raise ValueError('Segment transitions must be ordered inside (0, 1).')
    warped[specimen == name] = np.interp(position[specimen == name], knots_from, knots_to)
display(transitions.pivot(index='specimen', columns='transition', values='position').round(3))
warped_scores = pd.Series(cached_payload('warped_spatial', lambda: {'T_spatial': spatial_scores(all_rows, warped)},
                                         root=cache, inputs={**inputs, 'warped': warped}, code=code)['T_spatial'], index=genes)
warped_tests = joint_spatial('warped_matched', warped_scores)

### Region: cortical-like mouse S3, with a power-matched control

A mouse structure's depth is its mean distance to the three nearest reviewed glomeruli of the
same specimen, with centroids verified against the fine GeoJSON. Mouse S3 deeper than the 95th
percentile of the same specimen's S1 is removed, and everything is refit.

Removing those structures also removes power. So the same number of mouse S3 structures, drawn at
random from each mouse, is removed 20 times. A pathway is **region-sensitive** only if its
cortical effect ratio falls below the 5th percentile of these power-matched ratios, or turns
non-positive.

In [ ]:
from scipy.spatial import cKDTree
from pseudospace.repeated_inputs import _read_and_validate_segmentation


def mouse_depth():
    pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
    anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    pass1.file.close()
    values = np.full(len(position), np.nan)
    for name in expected['mouse']:
        native = ad.read_h5ad(data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad', backed='r')
        source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        native.file.close()
        mask = specimen == name
        structures = source_obs.loc[adata.obs_names[mask].str[len(name) + 1:]]
        _read_and_validate_segmentation(data_root, name, structures)
        glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
        _read_and_validate_segmentation(data_root, name, glomeruli)
        distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
            structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
        values[mask] = distance.mean(axis=1)
    return {'depth': values}


depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata.obs_names)}, code=code)['depth']
far_mouse_s3 = np.zeros(len(position), bool)
for name in expected['mouse']:
    limit = np.quantile(depth[(specimen == name) & (segment == 'PT-S1')], .95)
    far_mouse_s3 |= (specimen == name) & (segment == 'PT-S3') & (depth > limit)
cortical_keep = ~far_mouse_s3
print(f'Deep mouse S3 removed: {int(far_mouse_s3.sum()):,} structures '
      f'({far_mouse_s3[np.isin(specimen, expected["mouse"]) & (segment == "PT-S3")].mean():.0%} of mouse S3).')
cortical_scores = pd.Series(cached_payload('cortical_spatial', lambda: {'T_spatial': spatial_scores(cortical_keep, position)},
                                           root=cache, inputs={**inputs, 'keep': cortical_keep}, code=code)['T_spatial'], index=genes)
cortical_tests = joint_spatial('cortical_matched', cortical_scores)
power_effects = {}
for draw in range(N_POWER_DRAWS):
    rng = np.random.default_rng(SEED + draw)
    keep = np.ones(len(position), bool)
    for name in expected['mouse']:
        pool = np.flatnonzero((specimen == name) & (segment == 'PT-S3'))
        n_remove = int((far_mouse_s3 & (specimen == name)).sum())
        keep[rng.choice(pool, n_remove, replace=False)] = False
    draw_scores = pd.Series(cached_payload(f'power_spatial_{draw}', lambda keep=keep: {'T_spatial': spatial_scores(keep, position)},
                                           root=cache, inputs={**inputs, 'keep': keep}, code=code)['T_spatial'], index=genes)
    power_effects[draw] = joint_spatial(f'power_matched_{draw}', draw_scores).effect
power_effects = pd.DataFrame(power_effects)

### Gene-subset checks, finer matching and the trajectory-method annotation

- **Gene-subset checks.** Detected in both species, mouse sex-unbiased, flagged genes removed,
  and equal probe counts (hook). Each uses the original matching strata and its own family VIF.
- **Finer matching.** 5 quantile bins per covariate.
- **Trajectory method.** Notebook 12's earlier run on notebook 03's DPT coordinate, rescored with
  the joint test. This is an annotation only, and only meaningful for the default coordinate
  family.

In [ ]:
subset_tests = {}
for family in [f for f in families if f not in ('all', 'detection_5pct')]:
    subset_tests[family] = joint_spatial(f'{family}_matched', T_spatial, family=family)
subset_tests['matching_5bins'] = joint_spatial('matching_5bins_matched', T_spatial,
                                               strata_values=matching_strata(covariates, bins=5))
dpt_file = results_root / 'pt_pathway_remodeling' / 'pathway_spatial_evidence.csv'
dpt_tests = None
if dpt_file.is_file():
    dpt = pd.read_csv(dpt_file)
    dpt = dpt[dpt.statistic.eq('T_spatial')] if 'statistic' in dpt else dpt
    dpt_tests = joint_matched_test(dpt, dpt.set_index('pathway_id').variance_inflation, keys=()).set_index('pathway_id')

### The funnel, decisions, and comparison with the earlier 82/66/42 list

In [ ]:
original_effect = species_spatial.effect


def ratio(result):
    return (result.effect.reindex(original_effect.index) / original_effect).where(original_effect > 0)


checks = pd.DataFrame(index=species_spatial.index)
checks['ratio_registration'] = ratio(warped_tests)
checks['q_registration'] = warped_tests.q_joint.reindex(checks.index)
checks['ratio_cortical'] = ratio(cortical_tests)
checks['q_cortical'] = cortical_tests.q_joint.reindex(checks.index)
power_ratio = power_effects.reindex(checks.index).div(original_effect, axis=0)
checks['power_ratio_p05'] = power_ratio.quantile(.05, axis=1)
checks['power_ratio_median'] = power_ratio.median(axis=1)
for family, result in subset_tests.items():
    checks['ratio_' + family] = ratio(result)
    checks['q_' + family] = result.q_joint.reindex(checks.index)
checks['registration_pass'] = checks.ratio_registration.ge(CHECK_RATIO) & checks.q_registration.le(CHECK_Q)
checks['region_sensitive'] = ~(cortical_tests.effect.reindex(checks.index).gt(0)
                               & checks.ratio_cortical.ge(checks.power_ratio_p05))
for family in subset_tests:
    checks[family + '_pass'] = checks['ratio_' + family].ge(CHECK_RATIO) & checks['q_' + family].le(CHECK_Q)
checks['candidate'] = checks.index.isin(candidates)
checks['specimen_sensitive'] = checks.index.isin(sensitive)
checks['stable'] = checks.index.isin(stable)
checks['confident'] = checks.index.isin(confident)
checks['robust'] = checks.confident & checks.registration_pass & ~checks.region_sensitive
checks['core'] = checks.robust & checks.detected_in_both_pass & checks.sex_unbiased_pass
robust_ids = checks.index[checks.robust].tolist()
core_ids = checks.index[checks.core].tolist()
funnel = pd.Series({'tested': len(checks), 'joint-test candidates': int(checks.candidate.sum()),
                    'not specimen-sensitive': len(candidates - sensitive), 'stable (confident)': int(checks.confident.sum()),
                    'robust (registration + region)': len(robust_ids), 'core (+ shared-gene, sex-unbiased)': len(core_ids)})
funnel.rename('pathways').to_csv(tables / 'funnel.csv')
display(funnel.to_frame('pathways'))
relabel_hits = spatial.effect.gt(0) & spatial.q_joint.le(ALPHA)
relabel_joint = [int(relabel_hits[spatial.partition.eq(s)].sum()) for s in swaps]
E2_PRIMARY = max(relabel_joint) <= .05 * len(candidates) and len(robust_ids) >= 20
print(f'E2 decision: relabel joint calls {relabel_joint} vs {len(candidates)} species candidates; '
      f'{len(robust_ids)} robust → ' + ('the joint-test funnel is the primary list.' if E2_PRIMARY
                                        else 'keep the earlier list as primary and flag joint-test status.'))
old_file = results_root / 'pt_pathway_registration_sensitivity' / 'confident_pathways_with_robustness.csv'
old = pd.read_csv(old_file) if old_file.is_file() else pd.DataFrame(columns=['pathway_id', 'robustness', 'core'])
checks['old_confident82'] = checks.index.isin(old.pathway_id)
checks['old_robust66'] = checks.index.isin(old.pathway_id[old.robustness.eq('robust')])
checks['old_core42'] = checks.index.isin(old.pathway_id[old.core.astype(bool)])
display(pd.DataFrame({new: [int((checks[new] & checks[o]).sum()) for o in ('old_confident82', 'old_robust66', 'old_core42')]
                      for new in ('confident', 'robust', 'core')}, index=['in old 82', 'in old 66', 'in old 42']))

**Reading the funnel.** The joint test is applied to the relabelings and within-species
comparisons too. It therefore flags fewer pathways as specimen-sensitive than notebook 31's
matched test did, and some broad programs that notebook 31 excluded now pass. Table S2 gives each
pathway's **specificity margin**: the species joint z divided by the largest relabeled or
within-species joint z. Broad terms with a small margin should be read with that in mind.

## 9 · R3 · Programs and positional descriptors

**Programs.** Robust pathways are grouped by the genes that carry their signal: members in the top
10% of `T_spatial`. Grouping uses average linkage on 1 − overlap coefficient, cut at 0.5. Each
program is named after its strongest pathway and listed with its shared driver genes, which are
contributing genes present in at least half its pathways. This grouping is descriptive and made
after the list was frozen.

**Descriptors.** These are display rules on fitted curves, not tests:
- **D(s)**, the local divergence of the pathway's members against other genes, and where it peaks.
- **Direction at the peak**, from the median member Z.
- **Shape class** of the median member Δ(s): reversing, localized or graded.
- **Averaging loss**, the median over contributing members of 1 − |mean Δ| ÷ mean |Δ|: the share
  of the positional difference that a whole-PT average would cancel.

In [ ]:
contributing = set(T_spatial.index[T_spatial.ge(T_spatial.quantile(CONTRIBUTING_QUANTILE))])
programs = overlap_programs(gene_sets, robust_ids, contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id')
curves = local_pathway_curves(fit['z'], genes, {p: gene_sets[p] for p in robust_ids}, grid)
descriptions = spatial_descriptions(curves).set_index('pathway_id')
median_delta = pd.DataFrame({p: delta.loc[gene_sets[p]].median() for p in robust_ids}).T
gene_loss = pd.Series(averaging_loss(delta.to_numpy()), index=genes)
descriptors = pd.DataFrame(index=robust_ids)
descriptors['program'] = programs.program
descriptors['peak_position'] = descriptions.peak_position
descriptors['human_high_at_peak'] = descriptions.median_member_z_at_peak.gt(0)
descriptors['shape_class'] = [shape_class(median_delta.loc[p].to_numpy()) for p in robust_ids]
def signal_members(p):
    """Contributing members (top-decile T_spatial), or all members when fewer than three contribute."""
    chosen = [g for g in gene_sets[p] if g in contributing]
    return chosen if len(chosen) >= 3 else list(gene_sets[p])


descriptors['averaging_loss'] = [float(gene_loss.loc[signal_members(p)].median()) for p in robust_ids]
descriptors['contributing_members'] = [len(set(gene_sets[p]) & contributing) for p in robust_ids]


def program_table():
    rows = []
    for program, block in descriptors.groupby('program'):
        ids = block.index.tolist()
        lead = species_spatial.loc[ids, 'q_joint'].idxmin()
        counts_ = pd.Series([g for p in ids for g in signal_members(p)]).value_counts()
        shared = counts_[counts_ >= max(1, len(ids) / 2)].index
        drivers = T_spatial.loc[shared].sort_values(ascending=False).index[:8].tolist()
        rows.append({'program': program, 'name': pathway_names[lead], 'n_pathways': len(ids),
                     'n_core': int(checks.loc[ids, 'core'].sum()),
                     'direction': 'human-high' if block.human_high_at_peak.mean() >= .5 else 'mouse-high',
                     'median_peak': float(block.peak_position.median()), 'drivers': ', '.join(drivers),
                     'shape_classes': '; '.join(f'{k} {v}' for k, v in block.shape_class.value_counts().items()),
                     'pathways': '; '.join(pathway_names[ids])})
    return pd.DataFrame(rows).set_index('program')


program_summary = program_table()
display(program_summary.drop(columns='pathways'))
split = descriptors.groupby('human_high_at_peak').peak_position.agg(['size', 'median']).round(3)
display(split.rename(index={True: 'human-high at peak', False: 'mouse-high at peak'}))

## 10 · E3 · Competitive, donor-level external replication

**References.** Public multi-donor snRNA, summed per donor and segment group:
- **Census human cortex** (dataset 09b518f9, 6 donors): convoluted PT ("early") versus S3.
  Pre-specified as the primary human reference.
- **Lake et al. 2023 / KPMP human cortex** (CELLxGENE, version 1.5): donors with ≥ 50 nuclei in each
  of PT-S1, PT-S2 and PT-S3, healthy cortex only. S1 + S2 counts are pooled as "early". This
  reference was added at the coordinator's request after the protocol was fixed. It gets the same
  test and is reported beside the primary one.
- **Census mouse** (dataset 25818bf7): S1 + S2 versus S3. 12 male donors are the reference; 12
  female donors are a sex sensitivity.

Each donor gives Δ = log2CPM(S3) − log2CPM(early). Our own contrast uses reviewed segment labels,
so it does not depend on the coordinate: (human S3 − early) − (mouse S3 − early) from specimen
means.

**Explicit detrending.** Phase 1 found a global negative interaction shared by both datasets. Two
components are measured for each reference and reported (`table_S5_global_component.csv`) so they
can be compared with workstream 1's attenuation test:
- an **offset**: the median S3 − early contrast across informative genes, per donor or specimen;
- an **attenuation slope**: the interaction regressed on the mouse gradient. If human positions
  were noisier, the human gradient would shrink toward zero, and the interaction would become
  proportional to minus the mouse gradient.

The primary test removes the offset (per-donor and per-specimen median centring). The slope is
reported. The sensitivity removes it as well, from both sides: our interaction is replaced by its
residual from the fitted slope, and the external test compares human donors with mouse donors
scaled by (1 + slope). That sensitivity asks whether pathway differences **beyond** a uniform
flattening of human gradients replicate.

**Test, per reference.**
- Informative genes: detected in ≥ 10% of our structures in each species, and mean log2CPM ≥ 1
  in both external species.
- Per gene: t_ext is the Welch t of centred Δ, human donors against mouse donors.
- Per gene: a_g = sign(our centred interaction) × t_ext.
- Per pathway: a matched rank-AUC of a_g against **expression-matched** random genes (strata:
  external mean log2CPM, our detection, our coverage). The VIF comes from **donor-level**
  residual correlations, so donors are the replicates.
- BH within the robust list.

**Comparators.** Uncalled pathways (no species candidate call), pathways called only under
relabeling (specimen noise), and gene sets of expression-matched random genes for the gene-level
agreement. Pathways called only by the conventional screens get the analogous **average-level**
test: whole-PT human versus mouse, signed by our whole-PT DESeq2 direction.

In [ ]:
def specimen_segment_means():
    out = {}
    short = pd.Series(segment).str.replace('PT-', '', regex=False).to_numpy()
    for name in names:
        for label, mask in (('S3', short == 'S3'), ('early', np.isin(short, ['S1', 'S2']))):
            out[(name, label)] = np.asarray(Y[(specimen == name) & mask].mean(axis=0)).ravel()
    return pd.DataFrame(out, index=genes)


ours_means = specimen_segment_means()
ours_contrast = pd.DataFrame({name: ours_means[(name, 'S3')] - ours_means[(name, 'early')] for name in names})
ours_raw = ours_contrast[expected['human']].mean(axis=1) - ours_contrast[expected['mouse']].mean(axis=1)
mouse_gradient_ours = ours_contrast[expected['mouse']].mean(axis=1)


def census_donors(label, segment_map, early_parts):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_CELLS_EXTERNAL)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    late, early, whole, info = {}, {}, {}, []

    def log_cpm_of(index):
        return np.log2(table.loc[index].sum() / meta.loc[index, 'total_counts_all_genes'].sum() * 1e6 + 1)

    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        late_index = block.index[block.segment.eq('S3')]
        early_index = block.index[block.segment.isin(early_parts)]
        late[donor], early[donor] = log_cpm_of(late_index), log_cpm_of(early_index)
        whole[donor] = log_cpm_of(late_index.union(early_index))
        info.append({'donor': donor, 'sex': sex})
    frame = lambda values: pd.DataFrame(values).T.reindex(columns=genes)
    return frame(late), frame(early), frame(whole), pd.DataFrame(info).set_index('donor')


h_late, h_early, h_whole, h_info = census_donors('human', {
    'kidney proximal convoluted tubule epithelial cell': 'early', 'epithelial cell of proximal tubule segment 3': 'S3'}, ['early'])
m_late, m_early, m_whole, m_info = census_donors('mouse', {
    'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
    'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
males, females = m_info.index[m_info.sex.eq('male')], m_info.index[m_info.sex.eq('female')]

### Lake/KPMP cortex: per-donor pseudobulk

Raw counts are summed per donor × segment for healthy PT nuclei from cortex samples (`region == C`,
`tissue == cortex of kidney`), reading the matrix block by block. Genes are matched to our
ortholog map by Ensembl id, with the symbol as fallback, as in workstream 3's notebook 38. This
reuses its `csr_group_sums` helper.

In [ ]:
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'


def lake_pseudobulk():
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        keep = (obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3']) & obs.disease.eq('normal')
                & obs.region.eq('C') & obs.tissue.eq('cortex of kidney'))
        label = (obs.donor_id + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=genes, fill_value=0.)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy()}


lake_late = lake_early = None
if lake_file.is_file():
    lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                          params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'normal'},
                          inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': genes,
                                  'orthologs': digest(orthologs)}, code=code)
    lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'sex', 'segment'])
    lake_meta['nuclei'] = lake['nuclei']
    nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
    lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_CELLS_EXTERNAL).all(axis=1)]
    sums_frame = pd.DataFrame(lake['sums'], columns=genes)

    def lake_log_cpm(donor, parts):
        rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.isin(parts)]
        return np.log2(sums_frame.loc[rows].sum() / lake['totals'][rows].sum() * 1e6 + 1)

    lake_late = pd.DataFrame({d: lake_log_cpm(d, ['S3']) for d in lake_donors}).T
    lake_early = pd.DataFrame({d: lake_log_cpm(d, ['S1', 'S2']) for d in lake_donors}).T
    print(f'Lake/KPMP cortex donors with ≥ {MIN_CELLS_EXTERNAL} nuclei in S1, S2 and S3: {len(lake_donors)} '
          f'({lake_meta.drop_duplicates("donor").set_index("donor").loc[lake_donors, "sex"].value_counts().to_dict()})')
else:
    print(f'Lake/KPMP file not found ({lake_file}); the Lake reference is skipped.')

### Replication per reference

In [ ]:
def reference_analysis(stage, human_late, human_early, mouse_late, mouse_early, mouse_donors):
    """Gene- and pathway-level replication of our centred species × position interaction."""
    hd = human_late - human_early
    md = (mouse_late - mouse_early).loc[mouse_donors]
    level_h = pd.concat([human_late.mean(), human_early.mean()], axis=1).max(axis=1)
    level_m = pd.concat([mouse_late.loc[mouse_donors].mean(), mouse_early.loc[mouse_donors].mean()], axis=1).max(axis=1)
    keep = (covariates[['detection_mouse', 'detection_human']].min(axis=1).ge(.10) & level_h.ge(1) & level_m.ge(1)
            & hd.notna().all() & md.notna().all())
    g = keep.index[keep.to_numpy()]
    centred = ours_contrast - ours_contrast.loc[g].median()
    ours_c = centred[expected['human']].mean(axis=1) - centred[expected['mouse']].mean(axis=1)
    hc, mc = hd.sub(hd[g].median(axis=1), axis=0), md.sub(md[g].median(axis=1), axis=0)
    ext_raw, ext_c = hd.mean() - md.mean(), hc.mean() - mc.mean()
    t = pd.Series(welch_t(hc[g].to_numpy(), mc[g].to_numpy())[0], index=g)
    slope_ours = np.polyfit(mouse_gradient_ours.loc[g], ours_raw.loc[g], 1)
    ours_slope_free = ours_raw - np.polyval(slope_ours, mouse_gradient_ours)
    strata_g = matching_strata(pd.DataFrame({'mean_expression': ((level_h + level_m) / 2).loc[g].to_numpy(),
                                             'detection': covariates.loc[g, ['detection_mouse', 'detection_human']].min(axis=1).to_numpy(),
                                             'coverage': covariates.loc[g, 'coverage'].to_numpy()}), bins=3)
    sets_g = {p: [x for x in v if x in set(g)] for p, v in gene_sets.items()}
    sets_g = {p: v for p, v in sets_g.items() if 5 <= len(v) < len(g)}
    residuals = pd.concat([hc[g] - hc[g].mean(), mc[g] - mc[g].mean()])
    corr = residual_pathway_correlations(residuals.to_numpy(), np.repeat('donors', len(residuals)), list(g), sets_g)

    def pathway_test(statistic, name):
        inflation = correlation_adjusted_rank_tests(statistic, sets_g, corr).set_index('pathway_id').variance_inflation
        result = matched(f'{stage}_{name}', pd.DataFrame({'a': statistic}), sets_g, strata_g)
        return joint_matched_test(result, inflation, keys=()).set_index('pathway_id')

    # Uniform-flattening component removed on both sides: human gradient = (1 + beta) x mouse gradient.
    beta_ext = np.polyfit(md[g].mean(), ext_raw.loc[g], 1)[0]
    t_free = pd.Series(welch_t(hc[g].to_numpy(), (1 + beta_ext) * mc[g].to_numpy())[0], index=g)
    signed = pathway_test(np.sign(ours_c.loc[g]) * t, 'signed')
    slope_free = pathway_test(np.sign(ours_slope_free.loc[g]) * t_free, 'slope_free_both')
    robust_g = [p for p in robust_ids if p in signed.index]
    family_q_ = pd.Series(multipletests(signed.loc[robust_g, 'p_joint'], method='fdr_bh')[1], index=robust_g)
    replicated_ = family_q_.le(.10) & signed.loc[robust_g, 'effect'].gt(0)
    slope_q = pd.Series(multipletests(slope_free.loc[robust_g, 'p_joint'], method='fdr_bh')[1], index=robust_g)
    uncalled_ = [p for p in signed.index if p not in candidates]
    relabel_ = [p for p in relabel_called if p in signed.index]
    top = g[T_spatial.loc[g].ge(T_spatial.loc[g].quantile(.9)).to_numpy()]
    rng_ = np.random.default_rng(SEED)
    strata_series = pd.Series(strata_g, index=g)
    pools_ = {k: strata_series.index[strata_series.eq(k)] for k in strata_series.unique()}
    need_ = strata_series.loc[top].value_counts()
    random_rho_ = []
    for _ in range(1000):
        chosen = np.concatenate([rng_.choice(pools_[k], n, replace=False) for k, n in need_.items()])
        random_rho_.append(spearmanr(ours_c.loc[chosen], ext_c.loc[chosen]).statistic)
    by_stat = lambda score: spearmanr(ours_c.loc[g[score.loc[g].ge(score.loc[g].quantile(.9)).to_numpy()]],
                                      ext_c.loc[g[score.loc[g].ge(score.loc[g].quantile(.9)).to_numpy()]]).statistic
    summary = {
        'informative genes': len(g), 'human donors': len(hd), 'mouse donors': len(md),
        'robust pathways testable': len(robust_g), 'robust replicating (BH ≤ 0.10 in list)': int(replicated_.sum()),
        'fraction replicating': float(replicated_.mean()),
        'robust replicating, slope also removed': int((slope_q.le(.10) & slope_free.loc[robust_g, 'effect'].gt(0)).sum()),
        'median replication z, robust': float(signed.loc[robust_g, 'z_joint'].median()),
        'median replication z, uncalled': float(signed.loc[uncalled_, 'z_joint'].median()),
        'Mann–Whitney p, robust > uncalled': mannwhitneyu(signed.loc[robust_g, 'z_joint'], signed.loc[uncalled_, 'z_joint'],
                                                          alternative='greater').pvalue,
        'Mann–Whitney p, robust > relabel-called': (mannwhitneyu(signed.loc[robust_g, 'z_joint'], signed.loc[relabel_, 'z_joint'],
                                                                 alternative='greater').pvalue if relabel_ else np.nan),
        'gene ρ, all informative': spearmanr(ours_c.loc[g], ext_c.loc[g]).statistic,
        'gene ρ, top 10% T_spatial': spearmanr(ours_c.loc[top], ext_c.loc[top]).statistic,
        'gene ρ, expression-matched random (median)': float(np.median(random_rho_)),
        'gene ρ, expression-matched random (97.5th pct)': float(np.percentile(random_rho_, 97.5)),
        'gene ρ, top 10% T_level': by_stat(gene_scores['species'].T_level),
        'gene ρ, top 10% mean expression': by_stat(covariates.mean_expression),
        'sign agreement, top 10% T_spatial': float(np.mean(np.sign(ours_c.loc[top]) == np.sign(ext_c.loc[top])))}
    component = {'median interaction': float(ext_raw.loc[g].median()),
                 'fraction of genes negative': float((ext_raw.loc[g] < 0).mean()),
                 'median S3 − early, human': float(hd[g].mean().median()),
                 'median S3 − early, mouse': float(md[g].mean().median()),
                 'slope of interaction on mouse gradient': float(np.polyfit(md[g].mean(), ext_raw.loc[g], 1)[0]),
                 'R² of that slope': float(np.corrcoef(md[g].mean(), ext_raw.loc[g])[0, 1] ** 2)}
    ours_component = {'median interaction': float(ours_raw.loc[g].median()),
                      'fraction of genes negative': float((ours_raw.loc[g] < 0).mean()),
                      'median S3 − early, human': float(ours_contrast.loc[g, expected['human']].mean(axis=1).median()),
                      'median S3 − early, mouse': float(mouse_gradient_ours.loc[g].median()),
                      'slope of interaction on mouse gradient': float(slope_ours[0]),
                      'R² of that slope': float(np.corrcoef(mouse_gradient_ours.loc[g], ours_raw.loc[g])[0, 1] ** 2)}
    return {'genes': g, 'ours_c': ours_c, 'ext_c': ext_c, 't': t, 'tests': signed, 'slope_free': slope_free,
            'family_q': family_q_, 'replicated': replicated_, 'uncalled': uncalled_, 'relabel': relabel_, 'top': top,
            'summary': summary, 'component': component, 'ours_component': ours_component}


relabel_called = set(spatial.loc[spatial.partition.isin(swaps) & spatial.effect.gt(0) & spatial.q_normal.le(ALPHA), 'pathway_id'])
references = {'Census cortex − male mouse (primary)': ('ext_census_male', h_late, h_early, m_late, m_early, males),
              'Census cortex − female mouse': ('ext_census_female', h_late, h_early, m_late, m_early, females)}
if lake_late is not None:
    references['Lake/KPMP cortex − male mouse'] = ('ext_lake_male', lake_late, lake_early, m_late, m_early, males)
    references['Lake/KPMP cortex − female mouse'] = ('ext_lake_female', lake_late, lake_early, m_late, m_early, females)
replication_results = {label: reference_analysis(*spec) for label, spec in references.items()}
external_summary = pd.DataFrame({label: r['summary'] for label, r in replication_results.items()})
global_component = pd.DataFrame({'ours (Visium HD, segment means; primary gene set)':
                                 replication_results['Census cortex − male mouse (primary)']['ours_component'],
                                 **{label: r['component'] for label, r in replication_results.items()}})
external_summary.to_csv(tables / 'external_replication_summary.csv')
global_component.to_csv(tables / 'table_S5_global_component.csv')
display(global_component.round(3))
display(external_summary.round(4))

primary_result = replication_results['Census cortex − male mouse (primary)']
primary, info_genes, ours_c, ext_c = primary_result['tests'], primary_result['genes'], primary_result['ours_c'], primary_result['ext_c']
robust_ext, replicated, family_q = list(primary_result['replicated'].index), primary_result['replicated'], primary_result['family_q']
uncalled, top_spatial = primary_result['uncalled'], primary_result['top']
mw_uncalled = primary_result['summary']['Mann–Whitney p, robust > uncalled']
gene_agreement = pd.Series({'top 10% T_spatial': primary_result['summary']['gene ρ, top 10% T_spatial'],
                            'expression-matched random sets, median': primary_result['summary']['gene ρ, expression-matched random (median)']})
E3_PATHWAY_REPLICATION = replicated.mean() >= .5 and mw_uncalled <= .01
print('E3 decision (primary reference, pre-specified): ' + (
    'pathway-level external replication is supported.' if E3_PATHWAY_REPLICATION else
    'the pre-specified criterion is not met; R4 claims measurement-level agreement, with the pathway-level numbers reported.'))

**Reading E3.** Three findings, in order of how much they constrain the paper.

1. **Human gradients are uniformly flatter than mouse ones.** In every human dataset (ours and
   both external references), the human S3 − early contrast is a small fraction of the mouse
   contrast for most genes. The interaction is close to minus the mouse gradient: see the slope and
   R² in the global-component table. The external references never used our coordinate, so this is
   not an artefact of it. It is either human PT biology or a property shared by human segment
   annotations.
2. **Robust pathways replicate far better than uncalled ones** in every reference (Mann–Whitney
   rows), but the pre-specified 50% replication within the list must be checked against the
   primary reference (decision above).
3. **Beyond that flattening, little replicates.** Once the uniform flattening is removed from both
   sides, very few robust pathways still replicate. The screen therefore picks out the programs
   where mouse zonation is strongest and the human profile is flat. It does not show replicated,
   pathway-specific positional differences beyond that global pattern.

### Average-level test for pathways that only the conventional screens call

This uses whole-PT log2CPM per donor (Census human against male mouse), median-centred per donor.
Each gene's Welch t is signed by our whole-PT DESeq2 direction, and the matched test with a
donor-level VIF is applied. It asks whether conventional-only species pathways replicate their
**average** difference outside our cohort.

In [ ]:
conventional_hits = pd.concat([whole.assign(hit=whole.q.le(ALPHA))[['partition', 'pathway_id', 'hit']],
                               seg.assign(hit=seg.q.le(ALPHA))[['partition', 'pathway_id', 'hit']]])
conventional_species = set(conventional_hits.loc[conventional_hits.partition.eq('species') & conventional_hits.hit, 'pathway_id'])
whole_lfc = conventional_de[conventional_de.partition.eq('species') & conventional_de.scope.eq('whole_PT')].set_index('gene').log2FoldChange
h_wc = h_whole.sub(h_whole[info_genes].median(axis=1), axis=0)
m_wc = m_whole.loc[males].sub(m_whole.loc[males, info_genes].median(axis=1), axis=0)
t_main = pd.Series(welch_t(h_wc[info_genes].to_numpy(), m_wc[info_genes].to_numpy())[0], index=info_genes)
main_sets = {p: [x for x in v if x in set(info_genes)] for p, v in gene_sets.items()}
main_sets = {p: v for p, v in main_sets.items() if 5 <= len(v) < len(info_genes)}
main_residuals = pd.concat([h_wc[info_genes] - h_wc[info_genes].mean(), m_wc[info_genes] - m_wc[info_genes].mean()])
main_corr = residual_pathway_correlations(main_residuals.to_numpy(), np.repeat('donors', len(main_residuals)),
                                          list(info_genes), main_sets)
main_statistic = np.sign(whole_lfc.reindex(info_genes).fillna(0)) * t_main
main_strata = matching_strata(pd.DataFrame({'mean_expression': ((h_whole[info_genes].mean() + m_whole.loc[males, info_genes].mean()) / 2).to_numpy(),
                                            'detection': covariates.loc[info_genes, ['detection_mouse', 'detection_human']].min(axis=1).to_numpy(),
                                            'coverage': covariates.loc[info_genes, 'coverage'].to_numpy()}), bins=3)
main_effect = joint_matched_test(matched('external_main_effect', pd.DataFrame({'a': main_statistic}), main_sets, main_strata),
                                 correlation_adjusted_rank_tests(main_statistic, main_sets, main_corr).set_index('pathway_id').variance_inflation,
                                 keys=()).set_index('pathway_id')
conventional_only = sorted((conventional_species - candidates) & set(main_effect.index))
conv_others = [p for p in main_effect.index if p not in set(conventional_only)]
average_level = pd.Series({
    'conventional-only species pathways tested': len(conventional_only),
    'replicating their average difference (nominal p ≤ 0.05)': int(main_effect.loc[conventional_only, 'p_joint'].le(.05).sum()),
    'replicating (BH ≤ 0.10 within that list)': int((multipletests(main_effect.loc[conventional_only, 'p_joint'], method='fdr_bh')[1] <= .10).sum()),
    'median z, conventional-only': float(main_effect.loc[conventional_only, 'z_joint'].median()),
    'median z, other pathways': float(main_effect.loc[conv_others, 'z_joint'].median()),
    'Mann–Whitney p, conventional-only > other': mannwhitneyu(main_effect.loc[conventional_only, 'z_joint'],
                                                              main_effect.loc[conv_others, 'z_joint'], alternative='greater').pvalue})
average_level.to_csv(tables / 'external_average_level_conventional_only.csv')
display(average_level.to_frame('value'))

## 11 · E4 · Specimen-level split-plot test on positional pseudobulk

**Pseudobulk.** Raw counts are summed per specimen × coordinate bin: 8 equal-width bins on the
common support, keeping cells with at least 20 structures. Library size is the total over all
orthologs measured in both species.

**Model.** voom-style precision weights, then per gene a weighted linear model:
specimen + cubic B-spline(position, 3 df) + species × spline. The tested term is species ×
spline, judged against the specimen × position residual: 32 − 10 = 22 df when every cell is
present. Gene variances are moderated empirically as in limma.

**Relabelings** add the species × spline term as a nuisance and test the relabeled group ×
spline.

**Pathways.** Matched rank-AUC on the moderated F, with the VIF taken from **pseudobulk**
residual correlations. A self-contained pathway-score version runs as well: per-cell mean
z-scored member expression, tested with the same design by OLS F.

In [ ]:
from pseudospace.specimen import pseudobulk_profiles
from pseudospace.split_plot import log_cpm, voom_weights, moderated_f

inside = (position >= lo) & (position <= hi)
scaled = np.clip((position - lo) / (hi - lo), 0, 1)
stacked = sparse.hstack([counts, sparse.csr_matrix(library_size[:, None])], format='csr')
pseudobulk = cached_frame('pseudobulk_cells', lambda: pseudobulk_profiles(
    stacked[inside], specimen[inside], scaled[inside], n_bins=PSEUDOBULK_BINS, gene_names=genes + ['__library__'],
    min_structures_per_bin=MIN_STRUCTURES_PER_CELL), root=cache,
    params={'bins': PSEUDOBULK_BINS, 'min': MIN_STRUCTURES_PER_CELL}, inputs=inputs, code=code)
pb_counts = pseudobulk[['count_' + g for g in genes]].to_numpy(float)
pb_library = pseudobulk['count___library__'].to_numpy(float)
pb_specimen = pseudobulk.specimen.astype(str).to_numpy()
pb_human = np.isin(pb_specimen, expected['human']).astype(float)
pb_basis = make_gam_design(pseudobulk.pseudospace_bin_centre.to_numpy(float), np.array([]))[:, 1:]
pb_y = log_cpm(pb_counts, pb_library)
specimen_columns = (pb_specimen[:, None] == np.array(names)[None, :]).astype(float)
print(f'{len(pseudobulk)} specimen × bin cells (of {4 * PSEUDOBULK_BINS}); structures per cell '
      f'{int(pseudobulk.n_structures.min())}–{int(pseudobulk.n_structures.max())}')


from scipy.stats import f as f_distribution


def split_plot(label):
    if label == 'species':
        design = np.column_stack([specimen_columns, pb_basis, pb_human[:, None] * pb_basis])
    else:
        group = np.isin(pb_specimen, partitions[label]).astype(float)
        design = np.column_stack([specimen_columns, pb_basis, pb_human[:, None] * pb_basis, group[:, None] * pb_basis])
    test = list(range(design.shape[1] - pb_basis.shape[1], design.shape[1]))
    weights = voom_weights(pb_y, pb_library, design)
    result = moderated_f(pb_y, weights, design, test)
    scores = pd.DataFrame(pb_y - pb_y.mean(0), columns=genes) / np.maximum(pb_y.std(0), 1e-8)
    score_matrix = pd.DataFrame({p: scores[v].mean(axis=1) for p, v in gene_sets.items()})
    beta, *_ = np.linalg.lstsq(design, score_matrix.to_numpy(), rcond=None)
    reduced = np.delete(design, test, axis=1)
    beta_r, *_ = np.linalg.lstsq(reduced, score_matrix.to_numpy(), rcond=None)
    sse_full = ((score_matrix.to_numpy() - design @ beta) ** 2).sum(0)
    sse_reduced = ((score_matrix.to_numpy() - reduced @ beta_r) ** 2).sum(0)
    df_resid = design.shape[0] - design.shape[1]
    score_f = (sse_reduced - sse_full) / len(test) / (sse_full / df_resid)
    return {'F': result['F'], 'p': result['p'], 'df_total': np.array(min(result['d0'] + result['df_resid'], 1e6)),
            'd0': np.array(min(result['d0'], 1e6)), 'weighted_residuals': result['weighted_residuals'],
            'score_F': score_f, 'score_p': f_distribution.sf(score_f, len(test), df_resid)}


split_results = {label: cached_payload(stage_name('split_plot_', label), lambda label=label: split_plot(label),
                                       root=cache, inputs={'pb': digest(pb_y), 'library': pb_library}, code=code)
                 for label in partitions}
calibration_rows = {}
for label, result in split_results.items():
    p = result['p']
    expected_median = f_distribution.median(3, float(result['df_total']))
    calibration_rows[label] = {'genes p ≤ 0.01': float(np.mean(p <= .01)), 'lambda (median F / expected)': float(np.median(result['F']) / expected_median),
                               'genes BH ≤ 0.05': int((multipletests(p, method='fdr_bh')[1] <= .05).sum()), 'prior df d0': float(result['d0'])}
split_calibration = pd.DataFrame(calibration_rows).T
display(split_calibration.round(3))
pb_corr = residual_pathway_correlations(split_results['species']['weighted_residuals'], np.repeat('pb', len(pb_y)), genes, gene_sets)
pb_vif = correlation_adjusted_rank_tests(pd.Series(split_results['species']['F'], index=genes), gene_sets, pb_corr).set_index('pathway_id').variance_inflation
split_tests = pd.concat([joint_matched_test(matched(stage_name('split_matched_', label),
                                                    pd.DataFrame({'F_split': result['F']}, index=genes), gene_sets, strata),
                                            pb_vif, keys=()).assign(partition=label)
                         for label, result in split_results.items()], ignore_index=True)
split_calls = split_tests.assign(hit=lambda d: d.effect.gt(0) & d.q_joint.le(ALPHA)).groupby('partition').hit.sum()
split_scores_calls = {label: int((multipletests(r['score_p'], method='fdr_bh')[1] <= ALPHA).sum()) for label, r in split_results.items()}
split_species = split_tests[split_tests.partition.eq('species')].set_index('pathway_id')
split_z = split_tests.pivot(index='pathway_id', columns='partition', values='z_joint')
split_decoy = decoy_calls(split_z['species'], split_z[swaps], DECOY_ALPHA)
relabel_cal = split_calibration.loc[swaps]
split_ncdr = np.mean([split_calls[s] for s in swaps]) / max(split_calls['species'], 1)
score_ncdr = np.mean([split_scores_calls[s] for s in swaps]) / max(split_scores_calls['species'], 1)
calibrated = relabel_cal['lambda (median F / expected)'].max() <= 1.3 and relabel_cal['genes p ≤ 0.01'].max() <= .02
E4_DECISION = ('replace' if calibrated and split_ncdr <= .13 and split_calls['species'] >= 30 else
               'complement' if calibrated else
               'drop (limitation)' if relabel_cal['lambda (median F / expected)'].max() > 1.5 else 'complement with caveat')
split_summary = pd.Series({
    'species pathway calls (joint, BH ≤ 0.05)': int(split_calls['species']),
    **{f'relabel calls, {s.split("+")[-1]}': int(split_calls[s]) for s in swaps},
    'NCDR': split_ncdr, 'decoy calls at FDP ≤ 10%': len(split_decoy),
    'robust pathways also called (q ≤ 0.05)': int(split_species.reindex(robust_ids).pipe(lambda d: d.effect.gt(0) & d.q_joint.le(ALPHA)).sum()),
    'robust pathways also called (q ≤ 0.10)': int(split_species.reindex(robust_ids).pipe(lambda d: d.effect.gt(0) & d.q_joint.le(.10)).sum()),
    'gene Spearman, split-plot F vs T_spatial': spearmanr(split_results['species']['F'], T_spatial).statistic,
    'pathway-score variant: species calls': split_scores_calls['species'],
    'pathway-score variant: NCDR': score_ncdr})
split_summary.to_csv(tables / 'split_plot_summary.csv')
display(split_summary.to_frame('value'))
print(f'E4 decision: {E4_DECISION}')
checks['split_plot_q'] = split_species.q_joint.reindex(checks.index)
checks['ratio_split_plot'] = (split_species.effect.reindex(checks.index) / original_effect).where(original_effect > 0)

### Post-hoc diagnostics (not pre-specified)

In the joint test above, the inflation comes from pseudobulk residual correlations. Specimen ×
position deviations are shared across genes, so those correlations are larger than the
structure-level ones. Two diagnostics show whether this inflation is the binding constraint:
- the matched test without inflation (BH on its empirical p);
- the decoy-estimated inflation of the split-plot matched z, as in section 7.

They do not change the E4 decision.

In [ ]:
split_tests['matched_hit'] = split_tests.effect.gt(0) & split_tests.q_empirical.le(ALPHA)
split_matched_calls = split_tests.groupby('partition').matched_hit.sum()
split_null_z = split_tests[split_tests.partition.isin(swaps)].pivot(index='pathway_id', columns='partition', values='z_matched')
split_a, split_rho, _ = decoy_inflation(split_null_z, species_spatial.n_genes.reindex(split_null_z.index))
split_species_z = split_species.z_matched
uncalled_split = [p for p in split_species.index if p not in candidates]
split_diagnostics = pd.Series({
    'matched-only species calls (empirical q ≤ 0.05)': int(split_matched_calls['species']),
    **{f'matched-only relabel calls, {s.split("+")[-1]}': int(split_matched_calls[s]) for s in swaps},
    'matched-only calls among robust pathways': int((split_species.reindex(robust_ids).effect.gt(0)
                                                     & split_species.reindex(robust_ids).q_empirical.le(ALPHA)).sum()),
    'decoy inflation a': split_a, 'decoy inflation rho': split_rho,
    'median pseudobulk VIF': float(pb_vif.median()), 'median structure-level VIF': float(vif['all'].median()),
    'median split-plot z, robust': float(split_species_z.reindex(robust_ids).median()),
    'median split-plot z, uncalled': float(split_species_z.loc[uncalled_split].median()),
    'Mann–Whitney p, robust > uncalled (split-plot z)': mannwhitneyu(split_species_z.reindex(robust_ids),
                                                                     split_species_z.loc[uncalled_split], alternative='greater').pvalue})
split_diagnostics.to_csv(tables / 'split_plot_diagnostics.csv')
display(split_diagnostics.to_frame('value'))
checks['split_plot_matched_q'] = split_species.q_empirical.reindex(checks.index)

**Reading E4.**

- **Gene level.** The specimen-level split-plot test is calibrated under relabeling (fraction of
  genes at p ≤ 0.01 and λ in the calibration table). It nevertheless ranks genes quite
  differently from `T_spatial` (Spearman in the summary). Part of `T_spatial`'s gene ranking
  comes from structure-level precision, which the specimen-level test does not credit.
- **Pathway level.** The pre-specified joint test uses an inflation from pseudobulk residual
  correlations, and it calls nothing. The diagnostics show that this inflation is larger than the
  decoys support.
- **What still holds.** Without the inflation, the split-plot screen calls pathways that no
  relabeling produces, and the robust list's split-plot z lies far above that of uncalled
  pathways.

Following the pre-specified rule, E4 is a complement, not a replacement: Table S2 reports each
pathway's split-plot status. It does not change the reporting set.

## 12 · Figure 2 · R2 · Specificity of pathway strategies

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 8.5,
                     'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'svg.fonttype': 'none', 'pdf.fonttype': 42})
HUMAN, MOUSE, NULL, INK, ACCENT = '#D55E00', '#0072B2', '#9A9A9A', '#333333', '#009E73'


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=220)


fig = plt.figure(figsize=(14, 9.4))
ax = fig.add_axes([.05, .66, .17, .28])
signs = np.array([[-1, -1, 1, 1], [-1, 1, 1, -1], [-1, 1, -1, 1]], float) / 2
ax.imshow(signs, cmap=ListedColormap(['#9ecae1', '#fdae6b']), vmin=-.5, vmax=.5, aspect='auto')
for (i, j), v in np.ndenumerate(signs):
    ax.text(j, i, '+½' if v > 0 else '−½', ha='center', va='center', fontsize=9)
ax.set_xticks(range(4), ['M1', 'M2', 'H1', 'H2'])
ax.set_yticks(range(3), ['species', 'relabeling 1', 'relabeling 2'])
ax.set_title('a · The three balanced contrasts', loc='left')
note = fig.add_axes([.02, .50, .25, .12])
note.axis('off')
note.text(0, 1, 'Split-plot error terms\n'
          '• species offset (T_level): specimens within species, 2 df\n'
          '• species × position (T_spatial): specimen × position, 2(k−1) df\n'
          '• each relabeling = one draw of the section-level null\n'
          '• a term shared by H1 and H2 (donor, panel, region)\n'
          '  enters the species contrast and cancels in relabelings',
          fontsize=7.6, va='top', transform=note.transAxes)
ax = fig.add_axes([.52, .53, .30, .43])
show = specificity.drop(index=[i for i in specificity.index if i.endswith('(joint test)') and 'T_spatial)' not in i])
yy = np.arange(len(show))[::-1]
for y_value, (name, row) in zip(yy, show.iterrows()):
    relabel_values = [row[c] for c in show.columns if c.startswith('relabel_calls_')]
    ax.plot([min(relabel_values + [row.species_calls]), max(relabel_values + [row.species_calls])], [y_value] * 2,
            color='#DDDDDD', lw=1.2, zorder=1)
    for value, marker in zip(relabel_values, ('s', 'D')):
        ax.scatter(value, y_value, marker=marker, s=26, facecolor='white', edgecolor=NULL, zorder=2)
    color = HUMAN if row.specific_nb31_rule else INK
    ax.scatter(row.species_calls, y_value, s=40, color=color, zorder=3)
    ncdr = f'{row.ncdr:.2f}' if np.isfinite(row.ncdr) else '–'
    ax.text(1.01, y_value, f'NCDR {ncdr} · null {row.null_rate:.0%} · decoy-cal. {int(row.decoy_calls_fdp10)}',
            transform=ax.get_yaxis_transform(), va='center', fontsize=7, color=color)
ax.set_yticks(yy, show.index, fontsize=7.2)
ax.set_xscale('symlog', linthresh=10)
ax.set_xlim(left=-.5)
ax.set_xlabel('Pathways called (BH q ≤ 0.05)')
ax.set_title('b · Species split (filled; orange = specific) and both relabelings (open)', loc='left')
ax = fig.add_axes([.05, .09, .24, .32])
curve_styles = (('1 · Whole-PT pseudobulk + signed GSEA', 'whole-PT GSEA', INK, ':'),
                ('2 · S1/S2/S3 pseudobulk + signed GSEA', 'S1/S2/S3 GSEA', INK, '--'),
                ('4 · Gene GAM offset (T_level)', 'T_level', NULL, '-'),
                ('6 · Gene GAM × S1/S2/S3 steps', 'S1/S2/S3 steps', '#7f7f7f', '-.'),
                ('8 · Gene GAM × smooth position (T_spatial)', 'T_spatial, matched', '#F0A070', '-'),
                ('8 · Gene GAM × smooth position (T_spatial) (joint test)', 'T_spatial, joint', HUMAN, '-'))
for name, label, color, style in curve_styles:
    curve = decoy_curves[name]
    curve = curve[curve.target_calls >= 5]
    ax.plot(curve.target_calls, curve.fdp, color=color, ls=style, lw=1.6, label=label)
ax.axhline(DECOY_ALPHA, color='#999999', lw=.8)
ax.set(xscale='log', xlabel='Species pathways called (threshold lowered →)', ylabel='Decoy-estimated FDP', ylim=(0, 1.05))
ax.legend(frameon=False, fontsize=6.8, loc='upper center', bbox_to_anchor=(.5, -.2), ncol=3)
ax.set_title('c · Target–decoy false-discovery estimate', loc='left')
ax = fig.add_axes([.39, .09, .25, .32])
within = tests[tests.partition.str.startswith('within_') | tests.partition.eq('species')]
within = within[within.statistic.isin(['T_level', 'T_spatial'])].assign(hit=lambda d: d.effect.gt(0) & d.q_normal.le(ALPHA))
within_table = within.pivot_table(index='partition', columns='statistic', values='hit', aggfunc='sum')
within_table = within_table.loc[['species', 'within_mouse', 'within_human']].rename(
    index={'species': 'species split', 'within_mouse': 'mouse vs mouse', 'within_human': 'human vs human\nsection'})
within_table.to_csv(tables / 'within_species_controls.csv')
x = np.arange(len(within_table))
ax.bar(x - .19, within_table.T_level, .36, color='#BBBBBB', label='offset (T_level)')
ax.bar(x + .19, within_table.T_spatial, .36, color=HUMAN, label='smooth position (T_spatial)')
for xi, (a_value, b_value) in enumerate(within_table[['T_level', 'T_spatial']].to_numpy()):
    ax.text(xi - .19, a_value, int(a_value), ha='center', va='bottom', fontsize=7)
    ax.text(xi + .19, b_value, int(b_value), ha='center', va='bottom', fontsize=7)
ax.set_xticks(x, within_table.index, fontsize=7.5)
ax.set_ylabel('Pathways called (matched test)')
ax.legend(frameon=False, fontsize=7)
ax.set_title('d · Specimen differences are offsets', loc='left')
ax = fig.add_axes([.73, .09, .24, .32])
for stat, color in (('T_level', NULL), ('T_spatial', HUMAN)):
    z = tests[tests.statistic.eq(stat) & tests.partition.isin(swaps)].pivot(index='pathway_id', columns='partition', values='z_matched')
    sizes = species_spatial.n_genes.reindex(z.index)
    binned_mean = (z ** 2).mean(axis=1).groupby(pd.qcut(sizes, 6), observed=True).mean()
    ax.plot([interval.mid for interval in binned_mean.index], binned_mean.to_numpy(), 'o', color=color)
    m_grid = np.linspace(10, 300, 50)
    ax.plot(m_grid, inflation.loc[stat, 'a'] * (1 + (m_grid - 1) * inflation.loc[stat, 'rho']), color=color,
            label=f'{stat}: a = {inflation.loc[stat, "a"]:.2f}, ρ = {inflation.loc[stat, "rho"]:.3f}')
ax.axhline(1, color='#999999', lw=.8)
ax.set(xlabel='Pathway size (genes)', ylabel='Mean decoy z² (1 = calibrated)')
ax.legend(frameon=False, fontsize=7, loc='upper left')
ax.set_title('e · Specimen noise inflates offset screens', loc='left')
ax.text(.98, .04, f'DESeq2 genes (whole PT): species {int(de_counts.loc["species", "whole_PT"]):,};\n'
        f'relabelings {", ".join(str(int(de_counts.loc[s, "whole_PT"])) for s in swaps)}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=7)
save(fig, 'fig2_specificity')
plt.show()
plt.close(fig)

## 13 · Figure 3 · R3 · Positional atlas of robust pathways, grouped into programs

In [ ]:
order = descriptors.assign(prog_size=descriptors.program.map(descriptors.program.value_counts())).sort_values(
    ['prog_size', 'program', 'peak_position'], ascending=[False, True, True]).index
external_status = pd.Series('not testable', index=robust_ids)
external_status.loc[robust_ext] = np.where(replicated, 'replicates', 'not shown')
fig = plt.figure(figsize=(13, max(8, .16 * len(order) + 3.5)))
grid_spec = fig.add_gridspec(2, 3, width_ratios=[1.6, .55, 1], height_ratios=[.16 * len(order) + 1, 3.2], wspace=.35, hspace=.42)
ax = fig.add_subplot(grid_spec[0, 0])
rows = median_delta.loc[order]
limit = float(np.quantile(np.abs(rows.to_numpy()), .98))
image = ax.imshow(rows.clip(-limit, limit).to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit,
                  extent=(grid[0], grid[-1], len(rows), 0), interpolation='nearest')
ax.scatter(descriptors.loc[order, 'peak_position'], np.arange(len(order)) + .5, marker='|', s=26, color='black', lw=1.1)
ax.set_yticks(np.arange(len(order)) + .5, [('★ ' if checks.loc[p, 'core'] else '') + pathway_names[p][:44] for p in order], fontsize=6)
boundaries = np.flatnonzero(descriptors.loc[order, 'program'].to_numpy()[1:] != descriptors.loc[order, 'program'].to_numpy()[:-1]) + 1
for b in boundaries:
    ax.axhline(b, color='white', lw=2.2)
ax.set_xlabel('PT coordinate')
for s, (a_q, b_q) in {s: np.quantile(position[segment == s], [.1, .9]) for s in segments}.items():
    ax.text((a_q + b_q) / 2, -.6, s[-2:], ha='center', fontsize=7.5, color='#555555')
fig.colorbar(image, cax=ax.inset_axes([.2, -.09, .6, .015]), orientation='horizontal', label='Median member Δ, human − mouse (log-norm)')
ax.set_title(f'b · {len(order)} robust pathways, {descriptors.program.nunique()} programs (★ core; tick = peak)', loc='left', pad=14)
ax = fig.add_subplot(grid_spec[0, 1], sharey=None)
annotation = pd.DataFrame({'program': descriptors.loc[order, 'program'],
                           'external': external_status.loc[order].map({'replicates': 2, 'not shown': 1, 'not testable': 0}),
                           'split-plot': checks.loc[order, 'split_plot_matched_q'].le(ALPHA).astype(int)})
ax.imshow(np.column_stack([annotation.program % 2, annotation.external / 2, annotation['split-plot']]), aspect='auto',
          cmap='Greys', vmin=0, vmax=1.4, extent=(0, 3, len(order), 0), interpolation='nearest')
for yi, p in enumerate(order):
    ax.text(.5, yi + .5, int(descriptors.loc[p, 'program']), ha='center', va='center', fontsize=5.5,
            color='white' if descriptors.loc[p, 'program'] % 2 else 'black')
ax.set_xticks([.5, 1.5, 2.5], ['program', 'external', 'split-plot (matched)'], rotation=45, fontsize=7)
ax.set_yticks([])
ax.set_title('annotations', loc='left', fontsize=8, pad=14)
ax.text(0, -.26, 'external: dark = replicates (Census),\ngrey = not shown, white = not testable\nsplit-plot: dark = matched q ≤ 0.05', transform=ax.transAxes, fontsize=6, va='top')
right = grid_spec[0, 2].subgridspec(2, 1, hspace=.45)
ax = fig.add_subplot(right[0])
stages = funnel.iloc[1:]
ax.barh(np.arange(len(stages))[::-1], stages.values, color=['#BBBBBB'] * 3 + [HUMAN, '#8B3A00'], height=.6)
for y_value, value in zip(np.arange(len(stages))[::-1], stages.values):
    ax.text(value + 1, y_value, str(value), va='center', fontsize=8)
ax.set_yticks(np.arange(len(stages))[::-1], stages.index, fontsize=7.5)
ax.set_xlabel('Pathways')
ax.set_title(f'a · Joint-test funnel (of {funnel["tested"]:,} tested)', loc='left')
ax_inset = fig.add_subplot(right[1])
ax_inset.scatter(descriptors.peak_position, descriptions.loc[descriptors.index, 'median_member_z_at_peak'],
                 c=np.where(descriptors.human_high_at_peak, HUMAN, MOUSE), s=16, alpha=.8)
ax_inset.axhline(0, color='#999999', lw=.7)
ax_inset.set(xlabel='Peak position on PT coordinate', ylabel='Median member Z at peak')
ax_inset.set_title('c · Human-high programs peak early, mouse-high late', loc='left', fontsize=8.5)
top_programs = program_summary.sort_values('n_pathways', ascending=False).head(6).index
cards = grid_spec[1, :].subgridspec(2, 6, hspace=.6, wspace=.55)
bins_edges = np.linspace(lo, hi, 13)
bin_centres = (bins_edges[:-1] + bins_edges[1:]) / 2
binned_index = np.clip(np.searchsorted(bins_edges, position, side='right') - 1, 0, len(bin_centres) - 1)
for k, program in enumerate(top_programs):
    ids = descriptors.index[descriptors.program.eq(program)]
    ax = fig.add_subplot(cards[0, k])
    for p in ids:
        ax.plot(grid, median_delta.loc[p], color='#BBBBBB', lw=.7)
    ax.plot(grid, median_delta.loc[ids].mean(), color=HUMAN if program_summary.loc[program, 'direction'] == 'human-high' else MOUSE, lw=1.8)
    ax.axhline(0, color='#999999', lw=.6)
    ax.set_title(f'P{program} · {program_summary.loc[program, "name"][:22]}', fontsize=6.6, loc='left')
    ax.tick_params(labelsize=6)
    if k == 0:
        ax.set_ylabel('Δ(s)', fontsize=7)
    driver = program_summary.loc[program, 'drivers'].split(', ')[0]
    ax = fig.add_subplot(cards[1, k])
    i = gene_index.get_loc(driver)
    ax.plot(grid, fit['human'][i], color=HUMAN, lw=1.6)
    ax.plot(grid, fit['mouse'][i], color=MOUSE, lw=1.6)
    column = Y[:, i].toarray().ravel()
    for name in names:
        means = [column[(specimen == name) & (binned_index == b)].mean() if np.any((specimen == name) & (binned_index == b)) else np.nan
                 for b in range(len(bin_centres))]
        ax.scatter(bin_centres, means, s=5, color=HUMAN if name in expected['human'] else MOUSE, alpha=.7)
    ax.set_title(driver, fontsize=7.5, loc='left', style='italic')
    ax.tick_params(labelsize=6)
    if k == 0:
        ax.set_ylabel('log-norm expr.', fontsize=7)
fig.text(.01, .02, 'd · Six largest programs: member pathway Δ(s) (grey), program mean (colour), and the leading shared driver gene '
         '(fitted human orange, mouse blue; dots = specimen bin means).', fontsize=7.5)
save(fig, 'fig3_positional_programs')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(5.2, 3), layout='constrained')
for k, klass in enumerate(['graded', 'localized', 'reversing']):
    values = descriptors.loc[descriptors.shape_class.eq(klass), 'averaging_loss']
    ax.scatter(np.full(len(values), k) + np.random.default_rng(k).uniform(-.15, .15, len(values)), values, s=14,
               color=[HUMAN if h else MOUSE for h in descriptors.loc[values.index, 'human_high_at_peak']], alpha=.8)
    ax.text(k, 1.04, f'n = {len(values)}', ha='center', fontsize=7.5)
ax.set_xticks(range(3), ['graded', 'localized', 'reversing'])
ax.set_ylim(-.02, 1.1)
ax.set_ylabel('Averaging loss (share cancelled\nin a whole-PT mean)')
ax.set_title('e · What a whole-PT average loses', loc='left')
save(fig, 'fig3e_averaging_loss')
plt.show()
plt.close(fig)

## 14 · Figure 4 · R4 · Robustness and replication

In [ ]:
fig = plt.figure(figsize=(14, 8.8))
grid_spec = fig.add_gridspec(2, 3, width_ratios=[1.45, 1, 1], wspace=.45, hspace=.55)
ax = fig.add_subplot(grid_spec[:, 0])
check_rows = [('ratio_registration', 'anatomy-anchored coordinate'), ('ratio_cortical', 'cortical-like mouse S3 only'),
              ('ratio_detected_in_both', 'genes detected in both species'), ('ratio_sex_unbiased', 'mouse sex-biased genes removed'),
              ('ratio_flagged_removed', 'flagged genes removed'), ('ratio_matching_5bins', 'finer matching (5 bins)')]
check_rows += [('ratio_probe_equal', 'equal probe counts')] if 'ratio_probe_equal' in checks else []
retention = checks.loc[robust_ids]
dpt_ratio = None
if dpt_tests is not None:
    dpt_ratio = dpt_tests.effect.reindex(robust_ids) / original_effect.reindex(robust_ids)
    check_rows.append(('dpt', 'DPT coordinate (nb 03 / 12)'))
CORE_COLOR, ROBUST_COLOR = '#8B3A00', '#F4A261'
for yi, (column, label) in enumerate(check_rows):
    values = dpt_ratio if column == 'dpt' else retention[column]
    jitter = np.random.default_rng(yi).uniform(-.18, .18, len(values))
    ax.scatter(values, np.full(len(values), yi) + jitter, s=12, c=np.where(retention.core, CORE_COLOR, ROBUST_COLOR), alpha=.85)
    ax.text(1.02, yi, f'{(values >= CHECK_RATIO).mean():.0%} ≥ 0.7', transform=ax.get_yaxis_transform(), va='center', fontsize=7)
    if column == 'ratio_cortical':
        band = retention[['power_ratio_p05', 'power_ratio_median']].median()
        ax.plot([band.power_ratio_p05, band.power_ratio_median], [yi + .38] * 2, color=ACCENT, lw=3,
                label='power-matched removal (median p5 to median)')
ax.scatter([], [], color=CORE_COLOR, s=12, label='core')
ax.scatter([], [], color=ROBUST_COLOR, s=12, label='robust, not core')
ax.axvline(CHECK_RATIO, color='#999999', ls='--', lw=.8)
ax.axvline(1, color='#CCCCCC', lw=.8)
ax.set_yticks(range(len(check_rows)), [label for _, label in check_rows])
ax.set_xlabel('Effect retained (AUC effect ÷ original)')
ax.legend(frameon=False, fontsize=7, loc='upper center', bbox_to_anchor=(.5, -.07), ncol=2)
ax.set_title(f'a · Effect retention, {len(robust_ids)} robust pathways', loc='left')
ax = fig.add_subplot(grid_spec[0, 1])
table_t = transitions.pivot(index='specimen', columns='transition', values='position')
for name in names:
    ax.scatter(table_t.loc[name].values, [0, 1], color=HUMAN if name in expected['human'] else MOUSE,
               marker='o' if name in (names[0], names[2]) else 's', s=36, label=name)
ax.set_yticks([0, 1], table_t.columns)
ax.set(xlabel='Position on coordinate', ylim=(-.6, 1.6))
ax.legend(frameon=False, fontsize=6.5, loc='center right')
ax.set_title('b · Reviewed segment transitions', loc='left')
ax = fig.add_subplot(grid_spec[0, 2])
ax.scatter(ext_c.loc[info_genes], ours_c.loc[info_genes], s=3, color='#CCCCCC', rasterized=True, label='informative genes')
ax.scatter(ext_c.loc[top_spatial], ours_c.loc[top_spatial], s=5, color=INK, rasterized=True, label='top 10% T_spatial')
ax.axhline(0, color='#999999', lw=.6)
ax.axvline(0, color='#999999', lw=.6)
ax.set(xlabel='Census (human − male mouse)(S3 − early), centred', ylabel='Ours, centred')
ax.legend(frameon=False, fontsize=6.5, loc='upper left')
rho_text = '; '.join(f'{label.split(" ")[0]}: ρ {r["summary"]["gene ρ, top 10% T_spatial"]:.2f} vs matched '
                     f'{r["summary"]["gene ρ, expression-matched random (median)"]:.2f}'
                     for label, r in replication_results.items() if 'male mouse' in label and 'female' not in label)
ax.set_title('c · Gene-level agreement (top 10% T_spatial)', loc='left', fontsize=9)
ax.text(.02, .02, rho_text, transform=ax.transAxes, fontsize=6.6, va='bottom')
ax = fig.add_subplot(grid_spec[1, 1])
box_groups, box_labels, box_colors = [], [], []
for label, r in replication_results.items():
    if 'female' in label:
        continue
    short_label = label.split(' ')[0].replace('Lake/KPMP', 'Lake')
    robust_g = list(r['replicated'].index)
    box_groups += [r['tests'].loc[robust_g, 'z_joint'], r['tests'].loc[r['uncalled'], 'z_joint']]
    box_labels += [f'{short_label}\nrobust ({len(robust_g)})', f'{short_label}\nuncalled']
    box_colors += [HUMAN, '#BBBBBB']
box_groups.append(primary.loc[primary_result['relabel'], 'z_joint'])
box_labels.append(f'Census\nrelabel-called ({len(primary_result["relabel"])})')
box_colors.append(NULL)
ax.boxplot(box_groups, widths=.5, showfliers=False)
for k, (values, color) in enumerate(zip(box_groups, box_colors), start=1):
    many = len(values) > 200
    ax.scatter(np.full(len(values), k) + np.random.default_rng(k).uniform(-.12, .12, len(values)), values,
               s=3 if many else 9, alpha=.3 if many else .8, color=color)
ax.set_xticks(range(1, len(box_labels) + 1), box_labels, fontsize=6.5)
ax.axhline(0, color='#999999', lw=.7)
ax.set_ylabel('External replication z (donors as replicates)')
ax.set_title('d · Pathway-level replication', loc='left', fontsize=9)
ax = fig.add_subplot(grid_spec[1, 2])
for label, color in (('species', HUMAN), (swaps[0], NULL), (swaps[1], '#5f5f5f')):
    p = np.clip(np.sort(split_results[label]['p']), 1e-300, 1)
    expected_p = (np.arange(1, len(p) + 1) - .5) / len(p)
    ax.plot(-np.log10(expected_p), -np.log10(p), color=color, lw=1.4, label='species' if label == 'species' else 'relabeling')
ax.plot([0, 4.2], [0, 4.2], color='#999999', lw=.8)
ax.set(xlabel='Expected −log10 p', ylabel='Observed −log10 p', ylim=(0, 12))
ax.legend(frameon=False, fontsize=7)
ax.set_title(f'e · Split-plot gene test (relabel λ ≤ {relabel_cal["lambda (median F / expected)"].max():.2f})', loc='left', fontsize=9)
save(fig, 'fig4_robustness_replication')
plt.show()
plt.close(fig)

## 15 · Tables and run record

In [ ]:
table = pd.DataFrame(index=species_spatial.index)
table['name'] = pathway_names.reindex(table.index)
table['library'] = table.index.str.split('::').str[0]
table['n_tested'] = species_spatial.n_genes
for column in ('auc', 'effect', 'z_matched', 'vif', 'z_joint', 'p_joint', 'q_joint', 'q_normal', 'q_empirical'):
    table[column] = species_spatial[column]
table['max_null_z_joint'] = spatial[~spatial.partition.eq('species')].pivot(index='pathway_id', columns='partition', values='z_joint').max(axis=1)
table['specificity_margin'] = table.z_joint / table.max_null_z_joint.clip(lower=1)
table = table.join(checks.drop(columns=[c for c in checks if c in table]))
table = table.join(stability_summary[['n_retained', 'n_testable', 'retention_fraction']])
table = table.join(descriptors)
table['external_replication'] = external_status.reindex(table.index)
table = table.join(primary[['z_joint', 'p_joint']].rename(columns={'z_joint': 'external_z', 'p_joint': 'external_p'}))
table['external_q_within_robust'] = family_q.reindex(table.index)
table['conventional_species_call'] = table.index.isin(conventional_species)
table['leading_contributing_genes'] = [', '.join(T_spatial.loc[sorted(set(gene_sets[p]) & contributing)].sort_values(ascending=False).index[:8])
                                       for p in table.index]
table.sort_values(['core', 'robust', 'q_joint'], ascending=[False, False, True]).to_csv(tables / 'table_S2_pathways.csv')
pd.concat({k: r['tests'].reindex(robust_ids)[['effect', 'z_joint', 'p_joint']].assign(q_within_robust=r['family_q'].reindex(robust_ids))
           for k, r in replication_results.items()}, axis=1).to_csv(tables / 'table_S3_external_replication.csv')
program_summary.to_csv(tables / 'table_S4_programs.csv')
pd.DataFrame({'F_split': split_results['species']['F'], 'p_split': split_results['species']['p'], 'T_spatial': T_spatial,
              'external_t_census_male': primary_result['t'].reindex(genes), 'ours_interaction_centred': ours_c,
              'informative_external': pd.Series(genes, index=genes).isin(info_genes)},
             index=genes).to_csv(tables / 'gene_level_statistics.csv')
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'coordinate': COORDINATE_LABEL,
            'coordinate_file': str(coordinate_file.name) if coordinate_file else 'notebook 13 scFates (default)',
            'probe_counts': probe_status, 'n_structures': int(len(position)), 'n_genes': len(genes), 'n_pathways': len(gene_sets),
            'funnel': funnel.to_dict(), 'n_programs': int(descriptors.program.nunique()),
            'decisions': {'E1_decoy_fdp_primary': bool(E1_PRIMARY), 'E2_joint_funnel_primary': bool(E2_PRIMARY),
                          'E3_pathway_replication': bool(E3_PATHWAY_REPLICATION), 'E4': E4_DECISION},
            'external': external_summary.astype(str).to_dict(), 'average_level': average_level.astype(str).to_dict(),
            'split_plot': split_summary.astype(str).to_dict(),
            'caveat': 'Descriptive: two mice and two cortex sections from one male human donor. Relabeling measures '
                      'section-level noise only; donor-level noise is addressed only by external donors.'}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps({k: manifest[k] for k in ('funnel', 'n_programs', 'decisions')}, indent=2))